# Triumph speedometer — rotation training from epoch 47

**Start from the best epoch-47 model, train with full-circle dashboard rotations, and measure the change.**

1. Choose **Runtime → Change runtime type → T4 GPU**, then **Run all**.
2. Authorize the Google Drive mount when Colab asks. Dataset files download automatically from the public release.
3. Leave **`resume_latest`** selected to continue the newest completed epoch. Full training state is saved to Drive after every completed epoch.

Default experiment: `speedometer_rotation_v2_from47`. Its first run imports the verified best epoch-47 model into a **new dataset experiment**, resets optimizer/random-generator state once, and evaluates the source model before any updates. The new experiment has a fixed budget of **20 epochs, numbered 0 → 20**, subject to early stopping. Source epoch 47 remains in its provenance. Rerunning retains that target; it does not add another 20 epochs.

Later restarts restore model, optimizer, mixed-precision scaler, random generators, data-loader state, training history and early-stopping counters. Old runs remain intact. A disconnect during an epoch resumes from the last verified completed epoch.

The main dataset has **6,000 images: 4,800 training + 1,200 validation**, at 12 angles separated by 30°. Original 0° images are retained. All variants of a pose stay in one split. A separate **1,200-image robustness set** uses the 12 unseen offset angles 15°,45°,… and never affects model selection or early stopping. Before/after reports show 0°, rotated images and each angle separately. This tests a broad range of rotations; it does not establish perfect performance at every possible angle.

The **98% within ±2 km/h** target still needs independently labeled real footage. Validation and robustness images share the synthetic master and are not a real-camera test. The original 500-image notebook is preserved as `Speedometer_Baseline_500_Colab.ipynb`.

[Latest notebook in Colab](https://colab.research.google.com/github/sainadh812/triumph-speedometer-dataset/blob/main/Speedometer_Training_Colab.ipynb) · [Dataset release](https://github.com/sainadh812/triumph-speedometer-dataset/releases/tag/v2.0.0)

## 1. Runtime and training mode

- **`resume_latest`** restores the most recently published complete epoch, including its optimizer state.
- **`resume_best`** restores the complete epoch with the lowest validation MAE. This intentionally rewinds training state and retains the original target epoch.
- **`fresh`** initializes a new model from ImageNet weights, or random weights if `FRESH_PRETRAINED=False`. Use a **new experiment name**, for example `speedometer_fresh_v2`; that configuration is reused on later resumes.

Fresh experiments require no private epoch-47 checkpoint. They report after-training validation and robustness metrics, with no source-model before/after claim.

Existing experiments reuse their saved configuration; changing a fresh-only setting does not alter a resumed run. Exact continuation requires matching dependency versions, dataset, device type and training settings. Broken uploads are skipped only when a verified compatible recovery checkpoint exists. No valid checkpoint means an explicit error, never a silent fresh model.

In [ ]:
#@title Training mode and experiment
TRAIN_MODE = "resume_latest" #@param ["resume_latest", "resume_best", "fresh"]
EXPERIMENT_NAME = "speedometer_rotation_v2_from47" #@param {type:"string"}
FRESH_PRETRAINED = True #@param {type:"boolean"}
# Fresh requires a NEW experiment name; later resume that same name.
import os, sys, json, time, hashlib, zipfile, importlib.util, shutil
from pathlib import Path
import torch, torchvision
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Image as DisplayImage

IN_COLAB = importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
RUN_MODE = os.environ.get('SPEEDOMETER_RUN_MODE', 'pilot')
assert RUN_MODE in ('pilot', 'smoke')
TRAIN_MODE = os.environ.get('SPEEDOMETER_TRAIN_MODE', TRAIN_MODE)
EXPERIMENT_NAME = os.environ.get('SPEEDOMETER_EXPERIMENT', 'smoke_rotation_persistence_v2' if RUN_MODE == 'smoke' else EXPERIMENT_NAME)
IMPORT_PILOT_IF_MISSING = True
# Leave at 0 for normal resume. A positive value explicitly adds a NEW budget.
EXTEND_BY_EPOCHS = 0

WORK_DIR = Path(os.environ.get('SPEEDOMETER_WORK_DIR', '/content/speedometer_work' if IN_COLAB else str(Path.cwd() / 'speedometer_work')))
WORK_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = Path(os.environ.get('SPEEDOMETER_DATASET_ZIP', '/content/speedometer_bw_rotation_v2.zip' if IN_COLAB else str(Path.cwd() / 'speedometer_bw_rotation_v2.zip')))
DATASET_URL = 'https://github.com/sainadh812/triumph-speedometer-dataset/releases/download/v2.0.0/speedometer_bw_rotation_v2.zip'
EXPECTED_ZIP_SHA256 = 'ce4d6d102adac02603c0bdc964b0832d0ba661ec70e8401c6d4f8d195add2aa4'
EXPECTED_LABELS_SHA256 = 'a6e15644d5fa7098b5ea6ba1625b39b21c0c04c4a3d122356cc923dbf9aad4b6'
EXPECTED_ROBUSTNESS_LABELS_SHA256 = '4f6ebb8055d116d4f5522d48b6514df0e8c3e1968afd673530bc21122c621f1d'
EXPECTED_SEED_SHA256 = 'a284ecb0ceb4892074feea3802d9e79b9e4bc9892c25113b8934b5428a16e97c'
print('Mode:', RUN_MODE, TRAIN_MODE, '| torch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not connected')
if not hasattr(torch.amp, 'GradScaler'):
    raise RuntimeError('Use a current Colab PyTorch runtime with torch.amp.GradScaler.')
if RUN_MODE == 'pilot' and not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU and Run all again.')
if RUN_MODE == 'smoke':
    torch.set_num_threads(min(4, os.cpu_count() or 1))
    print('Local smoke check only: 128 pixels, 24/24 images, 2 epochs, random weights. Not an accuracy result.')

## 2. Mount Drive and load the implementation

Authorize Drive access so checkpoints can survive a runtime reset. Files are saved under **MyDrive/SpeedometerTraining/experiments/**. Each invocation receives a unique run folder. Local smoke mode uses a mock Drive directory and does not request Google authorization.

The complete implementation is embedded here. Inputs remain 512 × 512; grayscale is repeated into three channels and ImageNet-normalized. A spatial decoder predicts pivot/tip heatmaps, and a speed head predicts continuous km/h. Batch-normalization running statistics stay frozen. Existing image augmentations and labels are preserved.

In [ ]:
if RUN_MODE == 'smoke':
    DRIVE_ROOT = Path(os.environ.get('SPEEDOMETER_DRIVE_ROOT', str(WORK_DIR / 'mock_drive' / 'SpeedometerTraining')))
elif IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.ismount('/content/drive') or not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('Google Drive is not mounted. Check authorization and rerun; no temporary checkpoint folder was created.')
    DRIVE_ROOT = Path('/content/drive/MyDrive/SpeedometerTraining')
else:
    raise RuntimeError('The full pilot uses Colab with mounted Drive. Local mode is for smoke checks.')
SEED_CHECKPOINT = DRIVE_ROOT / 'experiments/speedometer_400_continuation_v1/runs/20261017_090632_970209_082ca611/best_model.pt'
print('Persistent experiment root:', DRIVE_ROOT)

In [ ]:
#@title Load embedded training and persistence modules
TRAINING_CORE_SOURCE = '"""Reproducible first speedometer training pilot (PyTorch / Google Colab).\n\nRun from Python:\n    result = train_experiment(TrainConfig(data_dir="/content/bw_dataset_500",\n                                         output_dir="/content/run_400"))\nThis evaluates synthetic validation data, never a real-world test set. Input\nimages already contain camera and appearance augmentation. No additional image\ngeometry is changed here, so keypoint targets remain aligned with the pixels.\n"""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nimport pickle\nimport random\nimport shutil\nimport tempfile\nimport time\nimport uuid\nfrom collections import Counter, defaultdict\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader, Dataset\nfrom torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small\n\n\n@dataclass\nclass TrainConfig:\n    data_dir: str\n    output_dir: str\n    input_size: int = 512\n    batch_size: int = 8\n    head_epochs: int = 5\n    finetune_epochs: int = 25\n    patience: int = 7\n    seed: int = 42\n    pretrained: bool = True\n    max_train_samples: int | None = None\n    max_val_samples: int | None = None\n    head_lr: float = 1e-3\n    finetune_lr: float = 1e-4\n    backbone_lr: float = 2e-5\n    weight_decay: float = 1e-4\n    num_workers: int = 0\n    amp: bool = True\n    device: str = "auto"\n    min_delta_kmh: float = 0.01\n    heatmap_weight: float = 0.2\n    coordinate_weight: float = 5.0\n    heatmap_sigma: float = 1.5\n\n\ndef _sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _json_write(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")\n\n\ndef _data_path(root, relative):\n    path = (Path(root) / relative).resolve()\n    if not path.is_relative_to(Path(root).resolve()):\n        raise ValueError(f"Dataset path is outside data_dir: {relative}")\n    return path\n\n\ndef verify_dataset(data_dir):\n    """Check every model input and label, plus pose-group split isolation."""\n    root = Path(data_dir)\n    label_path = root / "labels.json"\n    document = json.loads(label_path.read_text(encoding="utf-8"))\n    records = document["samples"]\n    if len(records) != document["count"]:\n        raise ValueError("labels.json sample count disagrees with metadata")\n    counts = Counter(row["dataset_split"] for row in records)\n    if dict(counts) != document["split_counts"]:\n        raise ValueError("Split counts disagree with labels.json metadata")\n    if set(counts) != {"train", "validation"}:\n        raise ValueError("Expected nonempty train and validation splits")\n    ids, paths, hashes, groups = set(), set(), set(), defaultdict(set)\n    for row in records:\n        sid, relative = row["sample_id"], row["image"]\n        if sid in ids or relative in paths:\n            raise ValueError(f"Duplicate sample ID or image path: {sid}")\n        ids.add(sid)\n        paths.add(relative)\n        image_path = _data_path(root, relative)\n        digest = _sha256(image_path)\n        if digest != row["image_sha256"]:\n            raise ValueError(f"Image checksum mismatch: {sid}")\n        if digest in hashes:\n            raise ValueError(f"Duplicate image content: {sid}")\n        hashes.add(digest)\n        with Image.open(image_path) as im:\n            if im.mode != "L" or im.size != (row["width"], row["height"]):\n                raise ValueError(f"Unexpected mode or dimensions: {sid}")\n        speed = float(row["speed_kmh"])\n        if not math.isfinite(speed) or not 0 <= speed <= 180:\n            raise ValueError(f"Invalid speed: {sid}")\n        for name in ("pivot", "tip"):\n            xy = np.asarray(row["needle_position_px"][name], dtype=float)\n            norm = np.asarray(row["needle_position_normalized"][name], dtype=float)\n            limits = np.array([row["width"] - 1, row["height"] - 1])\n            if xy.shape != (2,) or norm.shape != (2,):\n                raise ValueError(f"Invalid keypoint shape: {sid}/{name}")\n            if not np.isfinite(xy).all() or not np.all((xy >= 0) & (xy <= limits)):\n                raise ValueError(f"Out-of-frame keypoint: {sid}/{name}")\n            if not np.allclose(xy / limits, norm, atol=1e-6):\n                raise ValueError(f"Normalized keypoint mismatch: {sid}/{name}")\n        groups[row["group_id"]].add(row["dataset_split"])\n    leaking = [gid for gid, splits in groups.items() if len(splits) != 1]\n    if leaking:\n        raise ValueError(f"Pose groups cross splits: {leaking[:5]}")\n    source = document.get("source", {})\n    if source.get("file") and source.get("sha256"):\n        if _sha256(_data_path(root, source["file"])) != source["sha256"]:\n            raise ValueError("Approved source checksum mismatch")\n    report = {\n        "count": len(records), "split_counts": dict(counts),\n        "pose_groups": len(groups), "pose_group_leakage": False,\n        "all_image_hashes_verified": True, "labels_sha256": _sha256(label_path),\n        "source_sha256": source.get("sha256"),\n        "scope": "synthetic validation; no independent real test set",\n    }\n    return records, report\n\n\ndef select_group_subset(records, limit, seed):\n    """Deterministic nested subsets; never divide variants of a pose group.\n\n    Singleton approved examples precede shuffled paired groups. This makes the\n    100/250/400 learning-curve subsets nested and exact for this dataset.\n    A requested size that would divide a group is rounded down and reported.\n    """\n    groups = defaultdict(list)\n    for row in records:\n        groups[row["group_id"]].append(row)\n    ordered = sorted(groups)\n    rng = random.Random(seed)\n    rng.shuffle(ordered)\n    ordered.sort(key=lambda gid: len(groups[gid]))\n    limit = len(records) if limit is None else int(limit)\n    if limit <= 0:\n        raise ValueError("Subset size must be positive")\n    selected = []\n    for gid in ordered:\n        group = sorted(groups[gid], key=lambda row: row["sample_id"])\n        if len(selected) + len(group) > limit:\n            break\n        selected.extend(group)\n    if not selected:\n        raise ValueError("Subset limit is smaller than one complete pose group")\n    return selected\n\n\nclass DashboardDataset(Dataset):\n    def __init__(self, data_dir, records, input_size=512):\n        self.root, self.records, self.input_size = Path(data_dir), records, input_size\n\n    def __len__(self):\n        return len(self.records)\n\n    def __getitem__(self, index):\n        row = self.records[index]\n        image = preprocess_image(self.root / row["image"], self.input_size)\n        points = torch.tensor([row["needle_position_normalized"][name]\n                               for name in ("pivot", "tip")], dtype=torch.float32)\n        return {"image": image, "speed": torch.tensor(float(row["speed_kmh"])),\n                "points": points, "index": index}\n\n\ndef preprocess_image(image, input_size=512):\n    """Accept a PIL image or image path; return normalized 3-channel CHW input.\n\n    Grayscale is repeated into RGB solely to reuse ImageNet-pretrained weights.\n    The caller must crop real camera footage to match the training dashboard.\n    """\n    if isinstance(image, (str, Path)):\n        with Image.open(image) as opened:\n            gray = opened.convert("L")\n    elif isinstance(image, Image.Image):\n        gray = image.convert("L")\n    else:\n        raise TypeError("image must be a PIL image or a file path")\n    gray = gray.resize((input_size, input_size), Image.Resampling.BILINEAR)\n    array = np.asarray(gray, dtype=np.float32).copy() / 255.0\n    tensor = torch.from_numpy(array).unsqueeze(0).repeat(3, 1, 1)\n    mean = tensor.new_tensor([0.485, 0.456, 0.406])[:, None, None]\n    std = tensor.new_tensor([0.229, 0.224, 0.225])[:, None, None]\n    return (tensor - mean) / std\n\n\ndef _conv_block(in_channels, out_channels):\n    return nn.Sequential(nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),\n                         nn.GroupNorm(8, out_channels), nn.SiLU(inplace=True))\n\n\ndef heatmap_coordinates(logits):\n    """Spatial soft-argmax, returning normalized (x,y) in pivot/tip order."""\n    batch, channels, height, width = logits.shape\n    probabilities = logits.float().flatten(2).softmax(-1).view(batch, channels, height, width)\n    xs = torch.linspace(0, 1, width, device=logits.device)\n    ys = torch.linspace(0, 1, height, device=logits.device)\n    x = (probabilities.sum(dim=2) * xs).sum(-1)\n    y = (probabilities.sum(dim=3) * ys).sum(-1)\n    return torch.stack((x, y), dim=-1)\n\n\nclass SpeedometerModel(nn.Module):\n    """MobileNetV3-Small encoder, spatial keypoint decoder, continuous speed head."""\n    def __init__(self, pretrained=True):\n        super().__init__()\n        try:\n            encoder = mobilenet_v3_small(\n                weights=MobileNet_V3_Small_Weights.DEFAULT if pretrained else None)\n        except Exception as exc:\n            if pretrained:\n                raise RuntimeError("Pretrained MobileNet weights could not be loaded. "\n                                   "Enable internet and retry; no random-weight fallback was used.") from exc\n            raise\n        self.features = encoder.features\n        # torchvision MobileNetV3-Small skip channels: 16@1/4, 24@1/8, 48@1/16.\n        self.lateral = _conv_block(576, 96)\n        self.decode16 = _conv_block(96 + 48, 64)\n        self.decode8 = _conv_block(64 + 24, 32)\n        self.decode4 = _conv_block(32 + 16, 16)\n        self.heatmap_head = nn.Conv2d(16, 2, 1)\n        self.speed_head = nn.Sequential(nn.Linear(576 + 6, 128), nn.SiLU(),\n                                        nn.Dropout(0.1), nn.Linear(128, 1))\n\n    def forward(self, image):\n        skips = {}\n        value = image\n        for index, layer in enumerate(self.features):\n            value = layer(value)\n            if index in (1, 3, 8):\n                skips[index] = value\n        deep = value\n        value = self.lateral(deep)\n        for index, block in ((8, self.decode16), (3, self.decode8), (1, self.decode4)):\n            value = F.interpolate(value, size=skips[index].shape[-2:],\n                                  mode="bilinear", align_corners=True)\n            value = block(torch.cat((value, skips[index]), dim=1))\n        logits = self.heatmap_head(value)\n        points = heatmap_coordinates(logits)\n        relative = points[:, 1] - points[:, 0]\n        pooled = F.adaptive_avg_pool2d(deep, 1).flatten(1)\n        speed_input = torch.cat((pooled, points.flatten(1), relative), dim=1)\n        speed = self.speed_head(speed_input).squeeze(1).sigmoid() * 180.0\n        return {"speed": speed, "points": points, "heatmap_logits": logits}\n\n    def configure_stage(self, stage):\n        if stage not in {"heads", "finetune"}:\n            raise ValueError(f"Unknown stage: {stage}")\n        for param in self.features.parameters():\n            param.requires_grad_(False)\n        if stage == "finetune":\n            for layer in list(self.features.children())[-4:]:\n                for param in layer.parameters():\n                    param.requires_grad_(True)\n        self.train()\n\n    def train(self, mode=True):\n        super().train(mode)\n        # Small batches should not overwrite pretrained running BN statistics.\n        # Affine BN parameters may still train in the final unfrozen blocks.\n        for module in self.features.modules():\n            if isinstance(module, nn.modules.batchnorm._BatchNorm):\n                module.eval()\n        return self\n\n\ndef training_loss(output, batch, config):\n    logits = output["heatmap_logits"].float()\n    height, width = logits.shape[-2:]\n    points = batch["points"].float()\n    yy, xx = torch.meshgrid(torch.arange(height, device=logits.device),\n                            torch.arange(width, device=logits.device), indexing="ij")\n    centers = points * points.new_tensor([width - 1, height - 1])\n    squared = ((xx[None, None] - centers[:, :, 0, None, None]) ** 2\n               + (yy[None, None] - centers[:, :, 1, None, None]) ** 2)\n    target = torch.exp(-squared / (2 * config.heatmap_sigma ** 2)).flatten(2)\n    target = target / target.sum(-1, keepdim=True).clamp_min(1e-12)\n    heatmap = -(target * logits.flatten(2).log_softmax(-1)).sum(-1).mean()\n    coordinate = F.smooth_l1_loss(output["points"], points, beta=0.02)\n    # A unit is 10 km/h and the Huber transition is 2 km/h; gradients stay modest.\n    speed = F.smooth_l1_loss(output["speed"].float() / 10,\n                           batch["speed"].float() / 10, beta=0.2)\n    total = speed + config.heatmap_weight * heatmap + config.coordinate_weight * coordinate\n    return total, {"speed_loss": float(speed.detach()),\n                   "heatmap_loss": float(heatmap.detach()),\n                   "coordinate_loss": float(coordinate.detach())}\n\n\ndef _device(config):\n    return torch.device("cuda" if torch.cuda.is_available() else "cpu") if config.device == "auto" else torch.device(config.device)\n\n\ndef _seed_everything(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n\n\ndef _loader(dataset, config, shuffle=False):\n    generator = torch.Generator().manual_seed(config.seed)\n    return DataLoader(dataset, batch_size=config.batch_size, shuffle=shuffle,\n                      num_workers=config.num_workers, pin_memory=_device(config).type == "cuda",\n                      generator=generator, drop_last=False)\n\n\ndef metrics_from_rows(rows):\n    if not rows:\n        raise ValueError("Cannot evaluate an empty dataset")\n    errors = np.array([row["error_kmh"] for row in rows], dtype=float)\n    absolute = np.abs(errors)\n    return {\n        "n": len(rows), "pose_groups": len({row["group_id"] for row in rows}),\n        "mae_kmh": float(absolute.mean()),\n        "rmse_kmh": float(np.sqrt(np.mean(errors ** 2))),\n        "p95_absolute_error_kmh": float(np.percentile(absolute, 95)),\n        "max_absolute_error_kmh": float(absolute.max()),\n        "bias_kmh": float(errors.mean()),\n        **{f"within_{bound}_kmh_percent": float((absolute <= bound).mean() * 100)\n           for bound in (1, 2, 5)},\n        "pivot_mae_px": float(np.mean([row["pivot_error_px"] for row in rows])),\n        "tip_mae_px": float(np.mean([row["tip_error_px"] for row in rows])),\n        "keypoint_error_definition": "Mean Euclidean distance in original dataset-image pixels",\n        "scope": "synthetic validation only; not real-world test accuracy",\n    }\n\n\n@torch.inference_mode()\ndef evaluate_model(model, loader, device):\n    model.eval()\n    rows = []\n    for batch in loader:\n        output = model(batch["image"].to(device, non_blocking=True))\n        speeds = output["speed"].float().cpu().numpy()\n        points = output["points"].float().cpu().numpy()\n        for local, dataset_index in enumerate(batch["index"].tolist()):\n            record = loader.dataset.records[dataset_index]\n            predicted_speed = float(speeds[local])\n            row = {"sample_id": record["sample_id"], "group_id": record["group_id"],\n                   "parent_sample_id": record.get("parent_sample_id", record["sample_id"]),\n                   "rotation_degrees": float(record.get("rotation_degrees", 0.0)),\n                   "image": record["image"], "dataset_split": record["dataset_split"],\n                   "profile": record.get("augmentation", {}).get("photometric", {}).get("profile", "unknown"),\n                   "inferred_60_influence": bool(record.get("inferred_60_influence", False)),\n                   "target_speed_kmh": float(record["speed_kmh"]),\n                   "predicted_speed_kmh": predicted_speed,\n                   "error_kmh": predicted_speed - float(record["speed_kmh"])}\n            row["absolute_error_kmh"] = abs(row["error_kmh"])\n            for index, name in enumerate(("pivot", "tip")):\n                predicted = points[local, index] * np.array([record["width"] - 1, record["height"] - 1])\n                target = np.array(record["needle_position_px"][name])\n                row[f"{name}_error_px"] = float(np.linalg.norm(predicted - target))\n                for dimension, axis in enumerate(("x", "y")):\n                    row[f"predicted_{name}_{axis}"] = float(predicted[dimension])\n                    row[f"target_{name}_{axis}"] = float(target[dimension])\n            rows.append(row)\n    return metrics_from_rows(rows), rows\n\n\ndef rotation_metrics_from_rows(rows):\n    """Report added camera rotation in clockwise image degrees, never needle angle.\n\n    Repeated rotations of one parent/pose remain correlated validation examples;\n    per-angle metrics are diagnostics, not independent real-world test results.\n    """\n    angles = sorted({float(row.get("rotation_degrees", 0.0)) for row in rows})\n    by_angle = {\n        f"{angle:g}": metrics_from_rows([row for row in rows\n                                        if float(row.get("rotation_degrees", 0.0)) == angle])\n        for angle in angles\n    }\n    by_group = {}\n    for name, rotated in (("original_zero", False), ("rotated", True)):\n        members = [row for row in rows if (float(row.get("rotation_degrees", 0.0)) != 0.0) == rotated]\n        if members:\n            by_group[name] = metrics_from_rows(members)\n    return {"metrics_by_rotation_degrees": by_angle, "metrics_by_rotation_group": by_group,\n            "rotation_convention": "Additional whole-image rotation, positive clockwise; not needle angle"}\n\n\ndef _save_csv(path, rows):\n    if not rows:\n        return\n    with Path(path).open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef save_evaluation_artifacts(rows, history, data_dir, output_dir):\n    """Save inspectable predictions, hardest examples and training curves."""\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    out = Path(output_dir)\n    _save_csv(out / "validation_predictions.csv", rows)\n    _save_csv(out / "history.csv", history)\n    hardest = sorted(rows, key=lambda row: row["absolute_error_kmh"], reverse=True)[:12]\n    plot_rows = max(1, (len(hardest) + 3) // 4)\n    figure, axes = plt.subplots(plot_rows, 4, figsize=(16, 4.2 * plot_rows + 0.6), squeeze=False)\n    for axis in axes.flat:\n        axis.axis("off")\n    for axis, row in zip(axes.flat, hardest):\n        with Image.open(Path(data_dir) / row["image"]) as im:\n            axis.imshow(im, cmap="gray", vmin=0, vmax=255)\n        for prefix, color, style in (("target", "lime", "-"), ("predicted", "magenta", "--")):\n            axis.plot([row[f"{prefix}_pivot_x"], row[f"{prefix}_tip_x"]],\n                      [row[f"{prefix}_pivot_y"], row[f"{prefix}_tip_y"]],\n                      style, color=color, linewidth=1.5, marker="o", markersize=3)\n        axis.set_title(f"{row[\'sample_id\']} | target {row[\'target_speed_kmh\']:.1f}\\n"\n                       f"pred {row[\'predicted_speed_kmh\']:.1f} | error {row[\'error_kmh\']:+.1f} km/h", fontsize=10)\n    figure.suptitle("Hardest synthetic validation samples — green: label; magenta: prediction")\n    figure.tight_layout(rect=(0, 0, 1, 0.94))\n    figure.savefig(out / "worst_predictions.png", dpi=130)\n    plt.close(figure)\n    figure, axes = plt.subplots(1, 2, figsize=(11, 4))\n    axes[0].plot([h["epoch"] for h in history], [h["train_loss"] for h in history], label="Training loss")\n    axes[0].set(xlabel="Epoch", ylabel="Combined training loss")\n    axes[1].plot([h["epoch"] for h in history], [h["validation_mae_kmh"] for h in history])\n    axes[1].set(xlabel="Epoch", ylabel="Synthetic validation MAE (km/h)")\n    axes[1].ticklabel_format(axis="y", style="plain", useOffset=False)\n    for axis in axes:\n        axis.grid(alpha=0.25)\n    figure.tight_layout()\n    figure.savefig(out / "learning_curves.png", dpi=140)\n    plt.close(figure)\n\n\ndef _optimizer(model, config, stage):\n    heads = [p for name, p in model.named_parameters() if p.requires_grad and not name.startswith("features.")]\n    groups = [{"params": heads, "lr": config.head_lr if stage == "heads" else config.finetune_lr}]\n    backbone = [p for p in model.features.parameters() if p.requires_grad]\n    if backbone:\n        groups.append({"params": backbone, "lr": config.backbone_lr})\n    return torch.optim.AdamW(groups, weight_decay=config.weight_decay)\n\n\ndef train_experiment(config: TrainConfig):\n    """Train once, restore the best validation-MAE checkpoint and export reports.\n\n    max_*_samples and pretrained=False support CPU smoke tests; their metrics\n    must not be presented as trained model quality. Full runs use all 400/100.\n    """\n    if config.input_size < 64 or config.input_size % 32:\n        raise ValueError("input_size must be >= 64 and divisible by 32 (512 for the pilot)")\n    if min(config.head_epochs, config.finetune_epochs) < 0 or config.head_epochs + config.finetune_epochs < 1:\n        raise ValueError("At least one training epoch is required")\n    if config.batch_size < 1 or config.patience < 1 or config.heatmap_sigma <= 0:\n        raise ValueError("batch_size, patience and heatmap_sigma must be positive")\n    out = Path(config.output_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    if (out / "best_model.pt").exists():\n        raise FileExistsError("Use a new output_dir to preserve the previous experiment")\n    _seed_everything(config.seed)\n    records, integrity = verify_dataset(config.data_dir)\n    train_records = select_group_subset([r for r in records if r["dataset_split"] == "train"],\n                                         config.max_train_samples, config.seed)\n    val_records = select_group_subset([r for r in records if r["dataset_split"] == "validation"],\n                                       config.max_val_samples, config.seed)\n    integrity.update({"used_train_samples": len(train_records), "used_validation_samples": len(val_records),\n                      "used_train_ids": [r["sample_id"] for r in train_records],\n                      "used_validation_ids": [r["sample_id"] for r in val_records]})\n    _json_write(out / "data_integrity.json", integrity)\n    _json_write(out / "config.json", asdict(config))\n    train_loader = _loader(DashboardDataset(config.data_dir, train_records, config.input_size), config, True)\n    val_loader = _loader(DashboardDataset(config.data_dir, val_records, config.input_size), config)\n    device = _device(config)\n    model = SpeedometerModel(pretrained=config.pretrained).to(device)\n    amp_enabled = config.amp and device.type == "cuda"\n    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)\n    best_mae, best_epoch, epoch, history = float("inf"), 0, 0, []\n    start = time.time()\n    best_path = out / "best_model.pt"\n    print(f"Device: {device}; train={len(train_records)}, synthetic validation={len(val_records)}", flush=True)\n    for stage, epochs in (("heads", config.head_epochs), ("finetune", config.finetune_epochs)):\n        if not epochs:\n            continue\n        # Start fine-tuning from the best head-training checkpoint.\n        if stage == "finetune" and best_path.exists():\n            model.load_state_dict(torch.load(best_path, map_location=device, weights_only=True)["model_state_dict"])\n        model.configure_stage(stage)\n        optimizer = _optimizer(model, config, stage)\n        stale = 0\n        stage_best = float("inf")\n        for _ in range(epochs):\n            epoch += 1\n            model.train()\n            totals = Counter()\n            seen = 0\n            for batch in train_loader:\n                batch = {key: value.to(device, non_blocking=True) if isinstance(value, torch.Tensor) else value\n                         for key, value in batch.items()}\n                optimizer.zero_grad(set_to_none=True)\n                with torch.autocast(device_type=device.type, enabled=amp_enabled):\n                    output = model(batch["image"])\n                    loss, components = training_loss(output, batch, config)\n                if not torch.isfinite(loss):\n                    raise FloatingPointError("Nonfinite training loss; no result was silently accepted")\n                scaler.scale(loss).backward()\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 5.0)\n                scaler.step(optimizer)\n                scaler.update()\n                count = len(batch["speed"])\n                seen += count\n                totals["train_loss"] += float(loss.detach()) * count\n                for name, value in components.items():\n                    totals[name] += value * count\n            metrics, _ = evaluate_model(model, val_loader, device)\n            mae = metrics["mae_kmh"]\n            if not math.isfinite(mae):\n                raise FloatingPointError("Nonfinite validation MAE")\n            history.append({"epoch": epoch, "stage": stage,\n                            **{name: value / seen for name, value in totals.items()},\n                            "validation_mae_kmh": mae,\n                            "validation_within_2_kmh_percent": metrics["within_2_kmh_percent"],\n                            "validation_tip_mae_px": metrics["tip_mae_px"]})\n            print(f"Epoch {epoch:02d} {stage:8s} loss={totals[\'train_loss\']/seen:.4f} "\n                  f"val MAE={mae:.3f} km/h; within2={metrics[\'within_2_kmh_percent\']:.1f}%", flush=True)\n            # Save the actual minimum, separately from the patience improvement threshold.\n            if mae < best_mae:\n                best_mae, best_epoch = mae, epoch\n                torch.save({"format_version": 1, "model_state_dict": model.state_dict(),\n                            "config": asdict(config), "best_epoch": best_epoch,\n                            "validation_metrics": metrics, "labels_sha256": integrity["labels_sha256"],\n                            "keypoint_order": ["pivot", "tip"], "speed_range_kmh": [0, 180],\n                            "preprocessing": "PIL grayscale -> square bilinear resize -> repeated RGB -> ImageNet norm",\n                            "scope": "synthetic pilot; real camera validation pending"}, best_path)\n            if mae < stage_best - config.min_delta_kmh:\n                stage_best, stale = mae, 0\n            else:\n                stale += 1\n            _save_csv(out / "history.csv", history)\n            if stale >= config.patience:\n                print(f"Early stopping {stage}: no material MAE improvement for {stale} epochs", flush=True)\n                break\n    model.load_state_dict(torch.load(best_path, map_location=device, weights_only=True)["model_state_dict"])\n    metrics, rows = evaluate_model(model, val_loader, device)\n    profiles = {profile: metrics_from_rows([row for row in rows if row["profile"] == profile])\n                for profile in sorted({row["profile"] for row in rows})}\n    speed_bins = {}\n    for lower in range(0, 180, 20):\n        members = [row for row in rows if lower <= row["target_speed_kmh"] < lower + 20\n                   or (lower == 160 and row["target_speed_kmh"] == 180)]\n        if members:\n            speed_bins[f"{lower}_to_{lower + 20}_kmh"] = metrics_from_rows(members)\n    calibration_groups = {}\n    for influenced in (False, True):\n        members = [row for row in rows if row["inferred_60_influence"] == influenced]\n        if members:\n            calibration_groups["inferred_60_region" if influenced else "other_regions"] = metrics_from_rows(members)\n    report = {"metrics": metrics, "metrics_by_profile": profiles,\n              "metrics_by_speed_bin": speed_bins, "metrics_by_calibration_region": calibration_groups,\n              "speed_bin_convention": "20-km/h bins are lower-inclusive, upper-exclusive; final bin includes 180",\n              "best_epoch": best_epoch,\n              "epochs_run": epoch, "elapsed_seconds": time.time() - start, "device": str(device),\n              "pretrained": config.pretrained, "training_samples": len(train_records),\n              "validation_samples": len(val_records), "labels_sha256": integrity["labels_sha256"],\n              "torch_version": str(torch.__version__), "config": asdict(config),\n              "limitation": "Shared synthetic master: these are development validation results, "\n                            "not evidence of 98% real-world accuracy. No independent real test was run."}\n    _json_write(out / "metrics.json", report)\n    save_evaluation_artifacts(rows, history, config.data_dir, out)\n    return {**report, "checkpoint_path": str(best_path), "metrics_path": str(out / "metrics.json"),\n            "predictions_path": str(out / "validation_predictions.csv"),\n            "worst_predictions_path": str(out / "worst_predictions.png"),\n            "learning_curves_path": str(out / "learning_curves.png")}\n\n\ndef load_model(checkpoint_path, device="auto"):\n    """Load this module\'s tensor-only checkpoint without downloading weights."""\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if device == "auto" else torch.device(device)\n    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n    model = SpeedometerModel(pretrained=False)\n    model.load_state_dict(checkpoint["model_state_dict"])\n    model.to(device).eval()\n    return model, checkpoint\n\n\n@torch.inference_mode()\ndef predict_image(model, image, input_size=512):\n    """Predict a matched dashboard crop; coordinates refer to resized input.\n\n    No confidence score or unreadability detector is claimed by this model.\n    """\n    device = next(model.parameters()).device\n    model.eval()\n    tensor = preprocess_image(image, input_size).unsqueeze(0).to(device)\n    result = model(tensor)\n    points = result["points"][0].float().cpu().numpy()\n    return {"speed_kmh": float(result["speed"][0]),\n            "keypoints_normalized": {name: points[index].tolist() for index, name in enumerate(("pivot", "tip"))},\n            "keypoints_input_px": {name: (points[index] * (input_size - 1)).tolist()\n                                   for index, name in enumerate(("pivot", "tip"))},\n            "input_size": input_size}\n\n\n# Versioned training checkpoints deliberately do not change the model or pilot API.\n_ARCHITECTURE_ID = "mobilenet_v3_small_spatial_decoder_16_24_48_576_speed582_v1"\n_PREPROCESSING_ID = "PIL grayscale -> square bilinear resize -> repeated RGB -> ImageNet norm"\n_RESUME_CONFIG_KEYS = (\n    "input_size", "batch_size", "head_epochs", "finetune_epochs", "patience",\n    "seed", "head_lr", "finetune_lr", "backbone_lr", "weight_decay",\n    "num_workers", "amp", "min_delta_kmh", "heatmap_weight",\n    "coordinate_weight", "heatmap_sigma",\n)\n\n\ndef _persistent_data(config):\n    records, integrity = verify_dataset(config.data_dir)\n    train = select_group_subset([r for r in records if r["dataset_split"] == "train"],\n                                config.max_train_samples, config.seed)\n    validation = select_group_subset([r for r in records if r["dataset_split"] == "validation"],\n                                     config.max_val_samples, config.seed)\n    integrity = dict(integrity, used_train_samples=len(train), used_validation_samples=len(validation),\n                     used_train_ids=[r["sample_id"] for r in train],\n                     used_validation_ids=[r["sample_id"] for r in validation])\n    return train, validation, integrity\n\n\ndef _identity(config, integrity):\n    values = asdict(config)\n    return {\n        "architecture": _ARCHITECTURE_ID, "preprocessing": _PREPROCESSING_ID,\n        "labels_sha256": integrity["labels_sha256"],\n        "train_ids": integrity["used_train_ids"],\n        "validation_ids": integrity["used_validation_ids"],\n        "keypoint_order": ["pivot", "tip"], "speed_range_kmh": [0, 180],\n        "training_config": {key: values[key] for key in _RESUME_CONFIG_KEYS},\n        "device_type": _device(config).type,\n        "amp_enabled": bool(config.amp and _device(config).type == "cuda"),\n        "torch_version": str(torch.__version__),\n    }\n\n\ndef checkpoint_identity(config):\n    """Derive the strict experiment identity used for full-state resume.\n\n    Paths and initialization choice do not affect an already initialized model.\n    Dataset contents, exact subsets, training settings and runtime version do.\n    Switching runtime/library versions therefore fails explicitly rather than\n    presenting a different training trajectory as an exact continuation.\n    """\n    _, _, integrity = _persistent_data(config)\n    return _identity(config, integrity)\n\n\ndef _cpu_tree(value):\n    if isinstance(value, torch.Tensor):\n        return value.detach().cpu().clone()\n    if isinstance(value, dict):\n        return {key: _cpu_tree(item) for key, item in value.items()}\n    if isinstance(value, tuple):\n        return tuple(_cpu_tree(item) for item in value)\n    if isinstance(value, list):\n        return [_cpu_tree(item) for item in value]\n    return value\n\n\ndef _capture_rng():\n    numpy_state = np.random.get_state()\n    return {\n        "python": random.getstate(),\n        # NumPy arrays/classes are intentionally absent for weights_only=True.\n        "numpy": [str(numpy_state[0]), numpy_state[1].tolist(), int(numpy_state[2]),\n                  int(numpy_state[3]), float(numpy_state[4])],\n        "torch": torch.get_rng_state().clone(),\n        "cuda": [state.cpu().clone() for state in torch.cuda.get_rng_state_all()]\n                if torch.cuda.is_available() else [],\n        "cudnn_deterministic": bool(torch.backends.cudnn.deterministic),\n        "cudnn_benchmark": bool(torch.backends.cudnn.benchmark),\n    }\n\n\ndef _restore_rng(state):\n    random.setstate(state["python"])\n    np_state = state["numpy"]\n    np.random.set_state((np_state[0], np.asarray(np_state[1], dtype=np.uint32),\n                         np_state[2], np_state[3], np_state[4]))\n    torch.set_rng_state(state["torch"].cpu())\n    if state["cuda"]:\n        if not torch.cuda.is_available() or len(state["cuda"]) != torch.cuda.device_count():\n            raise ValueError("CUDA RNG state requires the same number of CUDA devices")\n        torch.cuda.set_rng_state_all(state["cuda"])\n    torch.backends.cudnn.deterministic = state["cudnn_deterministic"]\n    torch.backends.cudnn.benchmark = state["cudnn_benchmark"]\n\n\ndef _copy_verified(local_path, destination, expected_hash):\n    """Copy to an unpublished name, verify the destination bytes, then rename."""\n    destination = Path(destination)\n    partial = destination.with_name(destination.name + "." + uuid.uuid4().hex + ".partial")\n    with Path(local_path).open("rb") as source, partial.open("xb") as target:\n        shutil.copyfileobj(source, target, length=1024 * 1024)\n        target.flush()\n        os.fsync(target.fileno())\n    if _sha256(partial) != expected_hash:\n        raise IOError(f"Checkpoint copy verification failed: {partial}")\n    os.replace(partial, destination)\n    if _sha256(destination) != expected_hash:\n        raise IOError(f"Checkpoint verification after rename failed: {destination}")\n\n\ndef _save_durable(payload, destination, *, kind, identity, epoch, validation_mae_kmh):\n    """Publish completion only after local and destination verification.\n\n    Drive\'s FUSE mount cannot promise remote transactional fsync semantics. A\n    completion sidecar plus a fresh SHA256 check makes partial uploads ineligible\n    for discovery; prior epoch files remain independent recovery points.\n    """\n    destination = Path(destination)\n    with tempfile.TemporaryDirectory(prefix="speedometer_checkpoint_") as temporary:\n        staged = Path(temporary) / "checkpoint.pt"\n        with staged.open("wb") as handle:\n            torch.save(payload, handle)\n            handle.flush()\n            os.fsync(handle.fileno())\n        # Exercise the same safe loader used for restoration before publication.\n        torch.load(staged, map_location="cpu", weights_only=True)\n        digest = _sha256(staged)\n        _copy_verified(staged, destination, digest)\n        manifest = {\n            "manifest_version": 1, "kind": kind, "checkpoint": destination.name,\n            "sha256": digest, "size_bytes": destination.stat().st_size,\n            "epoch": int(epoch), "validation_mae_kmh": float(validation_mae_kmh),\n            "created_at_ns": time.time_ns(), "identity": identity,\n        }\n        sidecar = destination.with_suffix(".complete.json")\n        staged_manifest = Path(temporary) / "manifest.json"\n        _json_write(staged_manifest, manifest)\n        _copy_verified(staged_manifest, sidecar, _sha256(staged_manifest))\n    return {**manifest, "checkpoint_path": str(destination), "manifest_path": str(sidecar)}\n\n\ndef _load_verified(candidate):\n    path = Path(candidate["checkpoint_path"])\n    if path.stat().st_size != candidate["size_bytes"] or _sha256(path) != candidate["sha256"]:\n        raise IOError(f"Checkpoint changed or is incomplete: {path}")\n    payload = torch.load(path, map_location="cpu", weights_only=True)\n    required = {"model_state_dict", "optimizer_state_dict", "scaler_state_dict", "rng_state",\n                "loader_generator_states", "history", "training_state", "best_state", "identity"}\n    if (payload.get("format_version") != 2\n            or payload.get("checkpoint_kind") != "full_training_state"\n            or not required.issubset(payload)):\n        raise ValueError(f"Not a complete training-state checkpoint: {path}")\n    if payload["identity"] != candidate["identity"]:\n        raise ValueError(f"Manifest/checkpoint identity mismatch: {path}")\n    state = payload["training_state"]\n    state_keys = {"epoch", "stage", "stage_epoch", "stage_best", "stale", "target_epoch",\n                  "head_budget", "finetune_budget", "training_complete", "completion_reason"}\n    if not isinstance(state, dict) or not state_keys.issubset(state):\n        raise ValueError(f"Incomplete stage/counter state: {path}")\n    if state["stage"] not in {"heads", "finetune"}:\n        raise ValueError(f"Invalid stage: {path}")\n    if not {"epoch", "model_state_dict", "validation_metrics", "provenance"}.issubset(payload["best_state"]):\n        raise ValueError(f"Incomplete best-model state: {path}")\n    if not {"train", "validation"}.issubset(payload["loader_generator_states"]):\n        raise ValueError(f"Incomplete DataLoader RNG state: {path}")\n    if not {"python", "numpy", "torch", "cuda", "cudnn_deterministic", "cudnn_benchmark"}.issubset(payload["rng_state"]):\n        raise ValueError(f"Incomplete random state: {path}")\n    if not {"state", "param_groups"}.issubset(payload["optimizer_state_dict"]):\n        raise ValueError(f"Incomplete optimizer state: {path}")\n    if state["epoch"] != candidate["epoch"]:\n        raise ValueError(f"Manifest/checkpoint epoch mismatch: {path}")\n    if not math.isclose(payload["validation_metrics"]["mae_kmh"],\n                        candidate["validation_mae_kmh"], rel_tol=1e-12):\n        raise ValueError(f"Manifest/checkpoint metric mismatch: {path}")\n    return payload\n\n\ndef discover_checkpoints(search_root, *, expected_identity=None):\n    """Find hash-verified complete full-state checkpoints, skipping bad saves.\n\n    The returned list is sorted by publication time. A complete.json sidecar\n    never bypasses byte verification. Incompatible identities are not candidates.\n    An empty result is not permission to silently initialize a new model.\n    """\n    root = Path(search_root)\n    candidates = []\n    if not root.exists():\n        return candidates\n    for sidecar in root.rglob("epoch_*.complete.json"):\n        try:\n            manifest = json.loads(sidecar.read_text(encoding="utf-8"))\n            if not isinstance(manifest, dict):\n                continue\n            if manifest.get("manifest_version") != 1 or manifest.get("kind") != "full_training_state":\n                continue\n            if any(type(manifest.get(key)) is not int or manifest[key] < minimum\n                   for key, minimum in (("size_bytes", 1), ("epoch", 0), ("created_at_ns", 1))):\n                continue\n            digest = manifest.get("sha256")\n            if not isinstance(digest, str) or len(digest) != 64 or any(c not in "0123456789abcdef" for c in digest):\n                continue\n            if not isinstance(manifest.get("identity"), dict):\n                continue\n            if expected_identity is not None and manifest.get("identity") != expected_identity:\n                continue\n            filename = manifest["checkpoint"]\n            if Path(filename).name != filename:\n                continue\n            path = sidecar.parent / filename\n            if not path.resolve().is_relative_to(root.resolve()):\n                continue\n            if not math.isfinite(float(manifest["validation_mae_kmh"])):\n                continue\n            candidate = {**manifest, "checkpoint_path": str(path), "manifest_path": str(sidecar)}\n            _load_verified(candidate)\n            candidates.append(candidate)\n        except (OSError, ValueError, KeyError, TypeError, AttributeError, RuntimeError, EOFError,\n                pickle.UnpicklingError):\n            continue\n    return sorted(candidates, key=lambda item: (item["created_at_ns"], item["checkpoint_path"]))\n\n\ndef find_checkpoint(search_root, mode, expected_identity=None):\n    """Choose the newest save or the lowest-MAE epoch among compatible saves."""\n    if mode not in {"resume_latest", "resume_best"}:\n        raise ValueError("Checkpoint selection mode must be resume_latest or resume_best")\n    candidates = discover_checkpoints(search_root, expected_identity=expected_identity)\n    if not candidates:\n        raise FileNotFoundError("No complete, checksum-valid, compatible full-state checkpoint was found. "\n                                "Choose an explicit warm start/fresh run or repair the checkpoint location; "\n                                "training has not silently reset.")\n    if mode == "resume_latest":\n        return candidates[-1]\n    return min(candidates, key=lambda item: (item["validation_mae_kmh"], -item["created_at_ns"]))\n\n\ndef _validate_legacy(payload, config, identity):\n    if payload.get("format_version") != 1:\n        raise ValueError("warm_start expects a legacy format_version=1 best_model.pt")\n    if payload.get("labels_sha256") != identity["labels_sha256"]:\n        raise ValueError("Legacy checkpoint dataset labels do not match")\n    if payload.get("preprocessing") != _PREPROCESSING_ID:\n        raise ValueError("Legacy checkpoint preprocessing does not match")\n    if payload.get("keypoint_order") != ["pivot", "tip"] or payload.get("speed_range_kmh") != [0, 180]:\n        raise ValueError("Legacy checkpoint output definitions do not match")\n    legacy_config = payload["config"]\n    for key in ("input_size", "max_train_samples", "max_val_samples", "seed"):\n        if legacy_config.get(key) != getattr(config, key):\n            raise ValueError(f"Legacy checkpoint differs in {key}; warm start requires the same data selection")\n    if payload.get("best_epoch", 0) < 1:\n        raise ValueError("Legacy checkpoint has no valid best epoch")\n    if not math.isfinite(float(payload["validation_metrics"]["mae_kmh"])):\n        raise ValueError("Legacy checkpoint has no finite validation MAE")\n\n\ndef _validate_transfer(payload, config):\n    """Authorize a weights-only initialization, not a cross-dataset resume."""\n    if payload.get("format_version") != 1:\n        raise ValueError("warm_start_transfer expects an exported format_version=1 best_model.pt")\n    if payload.get("architecture") != _ARCHITECTURE_ID:\n        raise ValueError("Transfer checkpoint architecture identity does not match")\n    if payload.get("preprocessing") != _PREPROCESSING_ID:\n        raise ValueError("Transfer checkpoint preprocessing does not match")\n    if payload.get("keypoint_order") != ["pivot", "tip"] or payload.get("speed_range_kmh") != [0, 180]:\n        raise ValueError("Transfer checkpoint output definitions do not match")\n    if payload.get("config", {}).get("input_size") != config.input_size:\n        raise ValueError("Transfer checkpoint input_size differs from the new experiment")\n    if type(payload.get("best_epoch")) is not int or payload["best_epoch"] < 0:\n        raise ValueError("Transfer checkpoint has no valid source epoch")\n    source_hash = payload.get("labels_sha256")\n    if not isinstance(source_hash, str) or len(source_hash) != 64:\n        raise ValueError("Transfer checkpoint must identify its source dataset")\n    if not math.isfinite(float(payload.get("validation_metrics", {}).get("mae_kmh", float("nan")))):\n        raise ValueError("Transfer checkpoint has no finite source-validation MAE")\n\n\ndef _inference_payload(best, config, integrity, resume_metadata):\n    return {\n        "format_version": 1, "model_state_dict": best["model_state_dict"],\n        "config": asdict(config), "best_epoch": best["epoch"],\n        "validation_metrics": best["validation_metrics"],\n        "labels_sha256": integrity["labels_sha256"], "keypoint_order": ["pivot", "tip"],\n        "speed_range_kmh": [0, 180], "preprocessing": _PREPROCESSING_ID,\n        "architecture": _ARCHITECTURE_ID, "provenance": best["provenance"],\n        "resume_metadata": resume_metadata,\n        "scope": "synthetic validation; independent real camera validation pending",\n    }\n\n\ndef train_persistent(config: TrainConfig, *, mode="fresh", search_root=None,\n                     warm_start_path=None, additional_epochs=20,\n                     stop_after_epochs=None, extend_by_epochs=0):\n    """Train with immutable, verified per-epoch checkpoints on mounted Drive.\n\n    fresh ignores all checkpoints and uses config.pretrained for initialization.\n    warm_start imports legacy weights, explicitly resets the optimizer, and sets\n    target_epoch=legacy best_epoch+additional_epochs. resume_latest/resume_best\n    restore a full state, including its fixed target. Only extend_by_epochs adds\n    a new budget. Every invocation requires a new/empty branch output directory.\n\n    warm_start_transfer permits a new dataset/subset experiment with matching\n    model/input/output definitions. It copies weights only, starts epoch0 with\n    new optimizer/RNG state, and targets additional_epochs. Source epoch and old\n    validation metrics are provenance; a new-validation baseline is evaluated\n    before any update and preserved for best-model selection and comparison.\n\n    Resume is at an epoch boundary. Bitwise reproducibility still depends on the\n    device and deterministic kernels; state restoration never claims otherwise.\n    stop_after_epochs is a test/interruption hook, not an early-stopping policy.\n    """\n    if mode not in {"fresh", "warm_start", "warm_start_transfer", "resume_latest", "resume_best"}:\n        raise ValueError(f"Unknown training mode: {mode}")\n    if config.input_size < 64 or config.input_size % 32:\n        raise ValueError("input_size must be >=64 and divisible by 32")\n    if config.batch_size < 1 or config.patience < 1 or config.heatmap_sigma <= 0:\n        raise ValueError("batch_size, patience and heatmap_sigma must be positive")\n    if min(config.head_epochs, config.finetune_epochs, extend_by_epochs) < 0:\n        raise ValueError("Epoch budgets must be nonnegative")\n    if mode == "fresh" and config.head_epochs + config.finetune_epochs < 1:\n        raise ValueError("Fresh training needs at least one epoch")\n    if mode in {"warm_start", "warm_start_transfer"} and (warm_start_path is None or additional_epochs < 1):\n        raise ValueError("Warm starts require a weight checkpoint and positive additional_epochs")\n    if stop_after_epochs is not None and stop_after_epochs < 0:\n        raise ValueError("stop_after_epochs must be nonnegative")\n    if mode == "fresh" and stop_after_epochs == 0:\n        raise ValueError("A fresh run needs one completed epoch before it has a checkpoint")\n    if extend_by_epochs and mode not in {"resume_latest", "resume_best"}:\n        raise ValueError("extend_by_epochs is only for explicit resumed-budget extensions")\n    out = Path(config.output_dir)\n    if out.exists() and any(out.iterdir()):\n        raise FileExistsError("Choose a new empty output_dir; earlier checkpoints must be preserved")\n    train_records, val_records, integrity = _persistent_data(config)\n    identity = _identity(config, integrity)\n    candidate, restored = None, None\n    if mode.startswith("resume_"):\n        if search_root is None:\n            raise ValueError("Exact resume requires search_root")\n        candidate = find_checkpoint(search_root, mode, identity)\n        restored = _load_verified(candidate)\n    legacy, transfer = None, None\n    if mode == "warm_start":\n        legacy = torch.load(warm_start_path, map_location="cpu", weights_only=True)\n        _validate_legacy(legacy, config, identity)\n    elif mode == "warm_start_transfer":\n        transfer = torch.load(warm_start_path, map_location="cpu", weights_only=True)\n        _validate_transfer(transfer, config)\n    out.mkdir(parents=True, exist_ok=True)\n    _json_write(out / "data_integrity.json", integrity)\n    _json_write(out / "config.json", asdict(config))\n    _seed_everything(config.seed)\n    train_loader = _loader(DashboardDataset(config.data_dir, train_records, config.input_size), config, True)\n    val_loader = _loader(DashboardDataset(config.data_dir, val_records, config.input_size), config)\n    device = _device(config)\n    model = SpeedometerModel(pretrained=config.pretrained if mode == "fresh" else False).to(device)\n    amp_enabled = bool(config.amp and device.type == "cuda")\n    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)\n    best, history, latest_path, transfer_baseline = None, [], None, None\n    metadata = {"mode": mode, "source_checkpoint_path": None, "source_epoch": 0,\n                "optimizer_restored": False, "full_state_restored": False,\n                "initialization": "ImageNet" if config.pretrained else "random",\n                "initial_pretrained": bool(config.pretrained),\n                "extend_by_epochs": int(extend_by_epochs)}\n    state = {"epoch": 0, "stage": "heads" if config.head_epochs else "finetune",\n             "stage_epoch": 0, "stage_best": float("inf"), "stale": 0,\n             "head_budget": config.head_epochs, "finetune_budget": config.finetune_epochs,\n             "target_epoch": config.head_epochs + config.finetune_epochs,\n             "training_complete": False, "completion_reason": None}\n    metrics = None\n    if legacy is not None:\n        model.load_state_dict(legacy["model_state_dict"], strict=True)\n        epoch = int(legacy["best_epoch"])\n        metrics = legacy["validation_metrics"]\n        state.update(epoch=epoch, stage="finetune", target_epoch=epoch + int(additional_epochs),\n                     finetune_budget=int(additional_epochs))\n        metadata.update(source_checkpoint_path=str(Path(warm_start_path)), source_epoch=epoch,\n                        source_sha256=_sha256(warm_start_path), initialization="legacy trained weights",\n                        initial_pretrained=bool(legacy["config"]["pretrained"]),\n                        limitation="Legacy checkpoint has no optimizer/RNG state. Fine-tuning starts with a new optimizer.")\n        best = {"epoch": epoch, "model_state_dict": _cpu_tree(model.state_dict()),\n                "validation_metrics": metrics,\n                "provenance": {"kind": "legacy_baseline", "checkpoint_path": str(warm_start_path),\n                               "sha256": metadata["source_sha256"]}}\n    elif transfer is not None:\n        model.load_state_dict(transfer["model_state_dict"], strict=True)\n        source_epoch = int(transfer["best_epoch"])\n        state.update(epoch=0, stage="finetune", target_epoch=int(additional_epochs),\n                     head_budget=0, finetune_budget=int(additional_epochs))\n        metadata.update(\n            source_checkpoint_path=str(Path(warm_start_path)), source_epoch=source_epoch,\n            source_sha256=_sha256(warm_start_path), initialization="transferred trained weights",\n            initial_pretrained=bool(transfer.get("resume_metadata", {}).get(\n                "initial_pretrained", transfer["config"]["pretrained"])),\n            source_labels_sha256=transfer["labels_sha256"],\n            target_labels_sha256=integrity["labels_sha256"],\n            source_validation_metrics=transfer["validation_metrics"],\n            epoch_numbering="New experiment starts at0; source_epoch records prior training",\n            limitation="Weights-only transfer to a new experiment. Optimizer, RNG, history and early-stopping state reset.")\n        print(f"Evaluating source epoch{source_epoch} on the new validation set before any training update...",\n              flush=True)\n        metrics, baseline_rows = evaluate_model(model, val_loader, device)\n        if not math.isfinite(metrics["mae_kmh"]):\n            raise FloatingPointError("Nonfinite transfer baseline validation MAE")\n        baseline_report = {\n            "metrics": metrics, **rotation_metrics_from_rows(baseline_rows),\n            "source_epoch": source_epoch, "source_checkpoint_sha256": metadata["source_sha256"],\n            "source_labels_sha256": transfer["labels_sha256"],\n            "target_labels_sha256": integrity["labels_sha256"],\n            "validation_samples": len(val_records), "training_updates_before_evaluation": 0,\n        }\n        transfer_baseline = {"report": baseline_report, "rows": baseline_rows}\n        state["stage_best"] = metrics["mae_kmh"]\n        best = {"epoch": 0, "model_state_dict": _cpu_tree(model.state_dict()),\n                "validation_metrics": metrics,\n                "provenance": {"kind": "transferred_weights_baseline", "source_epoch": source_epoch,\n                               "checkpoint_path": str(warm_start_path), "sha256": metadata["source_sha256"],\n                               "source_labels_sha256": transfer["labels_sha256"],\n                               "evaluated_labels_sha256": integrity["labels_sha256"]}}\n    elif restored is not None:\n        model.load_state_dict(restored["model_state_dict"], strict=True)\n        state, best, history = dict(restored["training_state"]), restored["best_state"], restored["history"]\n        metrics = restored["validation_metrics"]\n        transfer_baseline = restored.get("transfer_baseline")\n        metadata.update(source_checkpoint_path=candidate["checkpoint_path"], source_epoch=state["epoch"],\n                        source_sha256=candidate["sha256"], optimizer_restored=True, full_state_restored=True,\n                        initial_pretrained=restored.get("resume_metadata", {}).get(\n                            "initial_pretrained", bool(restored["config"]["pretrained"])),\n                        initialization="full training state", parent_resume_metadata=restored.get("resume_metadata"))\n        latest_path = candidate["checkpoint_path"]\n        if extend_by_epochs:\n            state["target_epoch"] += int(extend_by_epochs)\n            state["finetune_budget" if state["finetune_budget"] else "head_budget"] += int(extend_by_epochs)\n            state.update(training_complete=False, completion_reason=None, stale=0)\n            # Explicit budget extension begins a new fine-tuning patience window.\n            state["stage_best"] = metrics["mae_kmh"]\n    model.configure_stage(state["stage"])\n    optimizer = _optimizer(model, config, state["stage"])\n    if restored is not None:\n        optimizer.load_state_dict(restored["optimizer_state_dict"])\n        scaler.load_state_dict(restored["scaler_state_dict"])\n        train_loader.generator.set_state(restored["loader_generator_states"]["train"])\n        val_loader.generator.set_state(restored["loader_generator_states"]["validation"])\n        _restore_rng(restored["rng_state"])\n    _json_write(out / "resume_metadata.json", metadata)\n    if transfer_baseline is not None:\n        _json_write(out / "baseline_validation_metrics.json", transfer_baseline["report"])\n        _save_csv(out / "baseline_validation_predictions.csv", transfer_baseline["rows"])\n    best_path = out / "best_model.pt"\n\n    def save_best():\n        _save_durable(_inference_payload(best, config, integrity, metadata), best_path,\n                      kind="inference_best", identity=identity, epoch=best["epoch"],\n                      validation_mae_kmh=best["validation_metrics"]["mae_kmh"])\n\n    def save_epoch(kind="epoch_end"):\n        nonlocal latest_path\n        filename = f"epoch_{state[\'epoch\']:05d}_{uuid.uuid4().hex[:12]}.pt"\n        payload = {\n            "format_version": 2, "checkpoint_kind": "full_training_state", "save_reason": kind,\n            "model_state_dict": _cpu_tree(model.state_dict()),\n            "optimizer_state_dict": _cpu_tree(optimizer.state_dict()),\n            "scaler_state_dict": _cpu_tree(scaler.state_dict()),\n            "rng_state": _capture_rng(),\n            "loader_generator_states": {"train": train_loader.generator.get_state().clone(),\n                                        "validation": val_loader.generator.get_state().clone()},\n            "history": list(history), "training_state": dict(state), "best_state": best,\n            "identity": identity, "config": asdict(config), "validation_metrics": metrics,\n            "resume_metadata": metadata, "labels_sha256": integrity["labels_sha256"],\n            "transfer_baseline": transfer_baseline,\n            "best_epoch": best["epoch"], "torch_version": str(torch.__version__),\n        }\n        saved = _save_durable(payload, out / filename, kind="full_training_state", identity=identity,\n                              epoch=state["epoch"], validation_mae_kmh=metrics["mae_kmh"])\n        latest_path = saved["checkpoint_path"]\n        # Pointers are conveniences only; discovery always validates epoch files.\n        _json_write(out / "latest_checkpoint.json", saved)\n\n    if best is not None:\n        save_best()  # The parent\'s best remains available even if this branch never improves.\n    if legacy is not None:\n        save_epoch("legacy_warm_start_boundary")\n    elif transfer is not None:\n        save_epoch("weights_transfer_baseline_boundary")\n    started, epochs_this_call = time.time(), 0\n    print(f"{mode}: device={device}; train={len(train_records)}, validation={len(val_records)}; "\n          f"epoch {state[\'epoch\']} -> target {state[\'target_epoch\']}", flush=True)\n    while (state["epoch"] < state["target_epoch"] and not state["training_complete"]\n           and (stop_after_epochs is None or epochs_this_call < stop_after_epochs)):\n        # Transition is performed after the prior boundary was saved. Restoring\n        # that boundary performs the same weight/optimizer transition exactly once.\n        if state["stage"] == "heads" and state["finetune_budget"] and (\n                state["stage_epoch"] >= state["head_budget"] or state["stale"] >= config.patience):\n            model.load_state_dict(best["model_state_dict"])\n            state.update(stage="finetune", stage_epoch=0, stage_best=float("inf"), stale=0)\n            model.configure_stage("finetune")\n            optimizer = _optimizer(model, config, "finetune")\n        model.train()\n        totals, seen = Counter(), 0\n        for batch in train_loader:\n            batch = {key: value.to(device, non_blocking=True) if isinstance(value, torch.Tensor) else value\n                     for key, value in batch.items()}\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type, enabled=amp_enabled):\n                output = model(batch["image"])\n                loss, components = training_loss(output, batch, config)\n            if not torch.isfinite(loss):\n                raise FloatingPointError("Nonfinite training loss; prior completed checkpoints remain available")\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 5.0)\n            scaler.step(optimizer)\n            scaler.update()\n            count = len(batch["speed"])\n            seen += count\n            totals["train_loss"] += float(loss.detach()) * count\n            for name, value in components.items():\n                totals[name] += value * count\n        metrics, _ = evaluate_model(model, val_loader, device)\n        mae = metrics["mae_kmh"]\n        if not math.isfinite(mae):\n            raise FloatingPointError("Nonfinite validation MAE")\n        state["epoch"] += 1\n        state["stage_epoch"] += 1\n        epochs_this_call += 1\n        history.append({"epoch": state["epoch"], "stage": state["stage"],\n                        **{name: value / seen for name, value in totals.items()},\n                        "validation_mae_kmh": mae,\n                        "validation_within_2_kmh_percent": metrics["within_2_kmh_percent"],\n                        "validation_tip_mae_px": metrics["tip_mae_px"]})\n        improved = best is None or mae < best["validation_metrics"]["mae_kmh"]\n        if improved:\n            best = {"epoch": state["epoch"], "model_state_dict": _cpu_tree(model.state_dict()),\n                    "validation_metrics": metrics,\n                    "provenance": {"kind": "training_epoch", "branch_directory": str(out),\n                                   "epoch": state["epoch"]}}\n        if mae < state["stage_best"] - config.min_delta_kmh:\n            state.update(stage_best=mae, stale=0)\n        else:\n            state["stale"] += 1\n        if state["epoch"] >= state["target_epoch"]:\n            state.update(training_complete=True, completion_reason="target_epoch_reached")\n        elif ((state["stage"] == "finetune" and state["stage_epoch"] >= state["finetune_budget"])\n              or (state["stage"] == "heads" and not state["finetune_budget"]\n                  and state["stage_epoch"] >= state["head_budget"])):\n            state.update(training_complete=True, completion_reason="stage_budget_reached")\n        elif state["stale"] >= config.patience and (state["stage"] == "finetune" or not state["finetune_budget"]):\n            state.update(training_complete=True, completion_reason="early_stopping")\n        # Full state is durable before summaries/plots or mutable inference views.\n        save_epoch()\n        if improved:\n            save_best()\n        _save_csv(out / "history.csv", history)\n        print(f"Epoch {state[\'epoch\']:02d} {state[\'stage\']:8s} loss={totals[\'train_loss\']/seen:.4f} "\n              f"val MAE={mae:.3f}; within2={metrics[\'within_2_kmh_percent\']:.1f}%; checkpoint saved", flush=True)\n        if stop_after_epochs is not None and epochs_this_call >= stop_after_epochs:\n            break\n    if best is None:\n        raise RuntimeError("No model was evaluated; no usable result exists")\n    model.load_state_dict(best["model_state_dict"])\n    evaluation, rows = evaluate_model(model, val_loader, device)\n    profiles = {profile: metrics_from_rows([r for r in rows if r["profile"] == profile])\n                for profile in sorted({r["profile"] for r in rows})}\n    bins = {}\n    for lower in range(0, 180, 20):\n        members = [r for r in rows if lower <= r["target_speed_kmh"] < lower + 20\n                   or (lower == 160 and r["target_speed_kmh"] == 180)]\n        if members:\n            bins[f"{lower}_to_{lower + 20}_kmh"] = metrics_from_rows(members)\n    regions = {}\n    for influenced in (False, True):\n        members = [r for r in rows if r["inferred_60_influence"] == influenced]\n        if members:\n            regions["inferred_60_region" if influenced else "other_regions"] = metrics_from_rows(members)\n    report = {\n        "metrics": evaluation, "metrics_by_profile": profiles, "metrics_by_speed_bin": bins,\n        **rotation_metrics_from_rows(rows),\n        "metrics_by_calibration_region": regions, "best_epoch": best["epoch"],\n        "epochs_run": state["epoch"], "epochs_this_call": epochs_this_call,\n        "target_epoch": state["target_epoch"], "training_complete": state["training_complete"],\n        "completion_reason": state["completion_reason"], "elapsed_seconds": time.time() - started,\n        "device": str(device), "pretrained": metadata["initial_pretrained"],\n        "training_samples": len(train_records), "validation_samples": len(val_records),\n        "labels_sha256": integrity["labels_sha256"], "torch_version": str(torch.__version__),\n        "config": asdict(config), "resume_metadata": metadata, "best_provenance": best["provenance"],\n        "baseline_validation_metrics": transfer_baseline["report"]["metrics"] if transfer_baseline else None,\n        "baseline_validation_report": transfer_baseline["report"] if transfer_baseline else None,\n        "baseline_predictions_path": str(out / "baseline_validation_predictions.csv") if transfer_baseline else None,\n        "baseline_metrics_path": str(out / "baseline_validation_metrics.json") if transfer_baseline else None,\n        "latest_checkpoint_path": str(latest_path), "checkpoint_path": str(best_path),\n        "metrics_path": str(out / "metrics.json"),\n        "predictions_path": str(out / "validation_predictions.csv"),\n        "worst_predictions_path": str(out / "worst_predictions.png"),\n        "learning_curves_path": str(out / "learning_curves.png"),\n        "speed_bin_convention": "20-km/h bins are lower-inclusive, upper-exclusive; final bin includes 180",\n        "limitation": "Shared synthetic master: development validation only, not evidence of 98% real accuracy.",\n    }\n    _json_write(out / "metrics.json", report)\n    save_evaluation_artifacts(rows, history, config.data_dir, out)\n    return report\n'
WORKFLOW_SOURCE = '"""Drive-backed experiment selection for the self-contained Colab notebook."""\nfrom dataclasses import asdict\nfrom datetime import datetime, timezone\nfrom importlib.metadata import version\nfrom pathlib import Path\nimport csv\nimport hashlib\nimport json\nimport os\nimport re\nimport uuid\n\n\nLEGACY_SHA256 = "a2ed22a3feb4d4d5f8552bc31af6c5051b66503840ab8a22d3f361a5e6a1dd36"\n\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef runtime_versions():\n    return {name: version(name) for name in ("torch", "torchvision", "numpy", "Pillow")}\n\n\ndef ensure_writable(directory):\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    probe = directory / (".write_probe_" + uuid.uuid4().hex)\n    try:\n        with probe.open("xb") as handle:\n            handle.write(b"speedometer drive write verification")\n            handle.flush()\n            os.fsync(handle.fileno())\n        if probe.read_bytes() != b"speedometer drive write verification":\n            raise IOError("Drive write/read verification failed")\n    finally:\n        probe.unlink(missing_ok=True)\n\n\ndef _save_manifest(path, document):\n    staged = path.with_name(path.name + "." + uuid.uuid4().hex + ".partial")\n    with staged.open("x", encoding="utf-8") as handle:\n        json.dump(document, handle, indent=2, allow_nan=False)\n        handle.flush()\n        os.fsync(handle.fileno())\n    if json.loads(staged.read_text(encoding="utf-8")) != document:\n        raise IOError("Experiment manifest verification failed")\n    os.replace(staged, path)\n\n\ndef epoch_artifacts(experiment_dir):\n    return sorted(path for path in Path(experiment_dir).rglob("epoch_*")\n                  if path.is_file() and (path.suffix == ".pt"\n                                        or path.name.endswith(".complete.json")\n                                        or ".partial" in path.name))\n\n\ndef require_clean_checkpoint_inventory(training, experiment_dir, identity):\n    """Use verified recovery states and preserve incomplete files for inspection."""\n    artifacts = epoch_artifacts(experiment_dir)\n    candidates = training.discover_checkpoints(experiment_dir, expected_identity=identity)\n    verified = {Path(item[key]).resolve() for item in candidates\n                for key in ("checkpoint_path", "manifest_path")}\n    invalid = [path for path in artifacts if path.resolve() not in verified]\n    if invalid and not candidates:\n        names = "\\n".join(str(path) for path in invalid[:6])\n        raise RuntimeError("Incomplete, corrupt, or incompatible epoch files exist. "\n                           "Training stopped; no fresh fallback or legacy reimport was attempted. "\n                           "Preserve the files and inspect/recover them before resuming:\\n" + names)\n    if invalid:\n        print(f"Warning: preserving and skipping {len(invalid)} incomplete/corrupt/incompatible epoch files.")\n        for path in invalid[:6]:\n            print("  Ignored:", path)\n        print("Resuming from a checksum-verified completed epoch instead.")\n    if artifacts and not candidates:\n        raise RuntimeError("Checkpoint artifacts exist but no verified full-state checkpoint is usable")\n    return candidates\n\n\ndef prepare_persistent_run(training, *, data_dir, drive_root, experiment_name,\n                           requested_mode="resume_latest", run_mode="pilot",\n                           fresh_pretrained=True, import_pilot_if_missing=True,\n                           warm_start_path=None, extend_by_epochs=0,\n                           core_sha256=None, first_import_mode="warm_start",\n                           approved_source_sha256=None, source_epoch=29,\n                           additional_epochs=20, train_sample_limit=400,\n                           smoke_train_samples=8, smoke_val_samples=4,\n                           dataset_name=None, allow_smoke_transfer=False):\n    """Resolve a new immutable run branch and the exact train_persistent call."""\n    approved_source_sha256 = LEGACY_SHA256 if approved_source_sha256 is None else approved_source_sha256\n    if requested_mode not in {"fresh", "resume_latest", "resume_best"}:\n        raise ValueError("Choose fresh, resume_latest, or resume_best")\n    if first_import_mode not in {"warm_start", "warm_start_transfer"}:\n        raise ValueError("The first import must be warm_start or warm_start_transfer")\n    if additional_epochs < 1:\n        raise ValueError("A warm start requires a positive fixed epoch budget")\n    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]{0,99}", experiment_name):\n        raise ValueError("Use a short experiment name containing letters, digits, _, - or .")\n    if run_mode not in {"pilot", "smoke"} or extend_by_epochs < 0:\n        raise ValueError("Invalid run mode or extension budget")\n    if requested_mode == "fresh" and extend_by_epochs:\n        raise ValueError("extend_by_epochs is only valid for explicit resumed-budget extensions")\n    root = Path(drive_root).resolve()\n    ensure_writable(root)\n    experiment = root / "experiments" / experiment_name\n    experiment.mkdir(parents=True, exist_ok=True)\n    manifest_path = experiment / "experiment_config.json"\n    artifacts = epoch_artifacts(experiment)\n    saved = None\n    if manifest_path.exists():\n        saved = json.loads(manifest_path.read_text(encoding="utf-8"))\n        if saved.get("manifest_version") != 1 or saved.get("experiment_name") != experiment_name:\n            raise ValueError("Experiment configuration manifest is invalid")\n    elif artifacts:\n        raise RuntimeError("Epoch saves exist without their experiment configuration. "\n                           "Restore the configuration manifest; do not create a new one over these saves.")\n    if requested_mode == "fresh" and (saved is not None or artifacts):\n        raise FileExistsError("Fresh training needs a new experiment name. Earlier runs were preserved. "\n                              "For example, use speedometer_fresh_v2; later choose resume_latest for that name.")\n    if saved is not None:\n        if saved["run_mode"] != run_mode:\n            raise ValueError("Smoke and pilot experiments must use different names")\n        if saved["runtime_versions"] != runtime_versions():\n            raise RuntimeError("Dependency versions differ from the saved experiment. Exact continuation "\n                               "requires its torch, torchvision, numpy and Pillow versions. "\n                               f"Saved: {saved[\'runtime_versions\']}; current: {runtime_versions()}. "\n                               "No new model was initialized.")\n        configuration = dict(saved["config"])\n    else:\n        use_legacy = requested_mode != "fresh"\n        smoke_transfer = (run_mode == "smoke" and allow_smoke_transfer\n                          and first_import_mode == "warm_start_transfer" and warm_start_path is not None)\n        if use_legacy and (not import_pilot_if_missing or (run_mode != "pilot" and not smoke_transfer)):\n            raise FileNotFoundError("No saved experiment exists. Choose fresh explicitly with a new "\n                                    "experiment name, or enable the verified legacy import for the pilot.")\n        if use_legacy:\n            legacy = Path(warm_start_path or (root / "pilot_best_epoch29.pt"))\n            if not legacy.is_file() or file_sha256(legacy) != approved_source_sha256:\n                raise ValueError(f"The approved epoch-{source_epoch} checkpoint is missing or has the wrong SHA256: "\n                                 + str(legacy) + ". No fresh model was initialized.")\n        configuration = asdict(training.TrainConfig(\n            data_dir=str(data_dir), output_dir="resolved_per_run",\n            input_size=128 if run_mode == "smoke" else 512,\n            batch_size=2 if run_mode == "smoke" else 8,\n            head_epochs=(0 if use_legacy else 1) if run_mode == "smoke" else (0 if use_legacy else 5),\n            finetune_epochs=(additional_epochs if use_legacy else 1) if run_mode == "smoke" else (additional_epochs if use_legacy else 25),\n            patience=7, seed=42,\n            pretrained=False if run_mode == "smoke" or use_legacy else bool(fresh_pretrained),\n            max_train_samples=smoke_train_samples if run_mode == "smoke" else train_sample_limit,\n            max_val_samples=smoke_val_samples if run_mode == "smoke" else None,\n            device="cpu" if run_mode == "smoke" else "auto",\n        ))\n        saved = {"manifest_version": 1, "experiment_name": experiment_name,\n                 "created_at_utc": datetime.now(timezone.utc).isoformat(), "run_mode": run_mode,\n                 "config": configuration, "runtime_versions": runtime_versions(),\n                 "initialization_kind": ("transfer_warm_start" if first_import_mode == "warm_start_transfer"\n                                          else "legacy_warm_start") if use_legacy else "fresh",\n                 "legacy_checkpoint": str(legacy) if use_legacy else None,\n                 "legacy_sha256": approved_source_sha256 if use_legacy else None,\n                 "first_import_mode": first_import_mode if use_legacy else None,\n                 "source_epoch": int(source_epoch) if use_legacy else None,\n                 "additional_epochs": int(additional_epochs) if use_legacy else 0,\n                 "dataset_name": dataset_name,\n                 "training_core_sha256_at_creation": core_sha256}\n    stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")\n    run_dir = experiment / "runs" / (stamp + "_" + uuid.uuid4().hex[:8])\n    configuration.update(data_dir=str(data_dir), output_dir=str(run_dir))\n    config = training.TrainConfig(**configuration)\n    identity = training.checkpoint_identity(config)\n    if saved.get("identity") is not None and saved["identity"] != identity:\n        raise RuntimeError("The saved experiment identity does not match this dataset, runtime, "\n                           "or training configuration. No weights were reset.")\n    candidates = require_clean_checkpoint_inventory(training, experiment, identity)\n    actual_mode = requested_mode\n    kwargs = {"mode": requested_mode, "search_root": str(experiment),\n              "additional_epochs": int(additional_epochs), "extend_by_epochs": int(extend_by_epochs)}\n    if requested_mode.startswith("resume_") and not candidates:\n        if saved["initialization_kind"] not in {"legacy_warm_start", "transfer_warm_start"} or not import_pilot_if_missing:\n            raise FileNotFoundError("This experiment has no completed full-state checkpoint. "\n                                    "Resume cannot silently restart it. Use a new name for a fresh experiment.")\n        legacy = Path(saved["legacy_checkpoint"])\n        if not legacy.is_file() or file_sha256(legacy) != saved["legacy_sha256"]:\n            raise ValueError("The saved experiment\'s approved legacy checkpoint is missing or changed")\n        if extend_by_epochs:\n            raise ValueError("Do not extend the budget on the initial legacy import")\n        actual_mode = saved.get("first_import_mode", "warm_start")\n        kwargs.update(mode=actual_mode, warm_start_path=str(legacy), additional_epochs=saved["additional_epochs"])\n    if not manifest_path.exists():\n        saved["identity"] = identity\n        _save_manifest(manifest_path, saved)\n    # train_persistent creates this unique branch only after input validation.\n    print("Experiment:", experiment)\n    print("Requested mode:", requested_mode, "| effective mode:", actual_mode)\n    if actual_mode == "warm_start":\n        start_epoch = saved.get("source_epoch", 29)\n        print(f"Verified legacy epoch {start_epoch} import: optimizer/RNG state is reset once.")\n        print(f"The fixed target is epoch {start_epoch + saved[\'additional_epochs\']}, subject to early stopping.")\n    elif actual_mode == "warm_start_transfer":\n        print(f"Verified source epoch {saved[\'source_epoch\']} imported into the new dataset experiment.")\n        print("Optimizer/RNG state resets once; source weights are evaluated before the first update.")\n        print(f"The NEW experiment runs epoch 0 -> {saved[\'additional_epochs\']}, subject to early stopping.")\n    elif actual_mode.startswith("resume_"):\n        selected = (candidates[-1] if actual_mode == "resume_latest" else\n                    min(candidates, key=lambda item: (item["validation_mae_kmh"], -item["created_at_ns"])))\n        print("Restoring complete state from:", selected["checkpoint_path"])\n        print("The saved target is retained; explicit extension:", extend_by_epochs)\n    else:\n        print("Fresh initialization:", "ImageNet" if config.pretrained else "random")\n    return {"config": config, "kwargs": kwargs, "experiment_dir": str(experiment),\n            "run_dir": str(run_dir), "manifest_path": str(manifest_path),\n            "requested_mode": requested_mode, "actual_mode": actual_mode}\n\n\ndef read_prediction_rows(path):\n    """Read exported evaluation rows with numeric fields restored."""\n    with Path(path).open(newline="", encoding="utf-8") as handle:\n        rows = list(csv.DictReader(handle))\n    numeric = {"target_speed_kmh", "predicted_speed_kmh", "error_kmh", "absolute_error_kmh",\n               "pivot_error_px", "tip_error_px", "rotation_degrees"}\n    for row in rows:\n        for key in numeric.intersection(row):\n            row[key] = float(row[key])\n        row.setdefault("rotation_degrees", 0.0)\n    return rows\n\n\ndef rotation_breakdown(training, rows):\n    """Metrics grouped by known image rotation, with no effect on selection."""\n    result = {"all": training.metrics_from_rows(rows)}\n    for label, predicate in (("original_zero", lambda angle: abs(angle) < 1e-8),\n                             ("rotated", lambda angle: abs(angle) >= 1e-8)):\n        members = [row for row in rows if predicate(float(row.get("rotation_degrees", 0)))]\n        if members:\n            result[label] = training.metrics_from_rows(members)\n    for angle in sorted({float(row.get("rotation_degrees", 0)) for row in rows}):\n        members = [row for row in rows if float(row.get("rotation_degrees", 0)) == angle]\n        result[f"angle_{angle:g}"] = training.metrics_from_rows(members)\n    return result\n\n\ndef evaluate_rotation_checkpoint(training, *, checkpoint_path, data_dir, output_dir,\n                                 label_filename="labels.json", expected_labels_sha256=None,\n                                 batch_size=8, max_samples=None, device="auto"):\n    """Evaluate a checkpoint on a validation-only manifest, including robustness.\n\n    This path never calls an optimizer, early stopping, or model selection.\n    Image and label hashes are rechecked; cached results are bound to exact\n    checkpoint/label/sample identities rather than reused by filename alone.\n    """\n    import numpy as np\n    import torch\n    from PIL import Image\n    from torch.utils.data import DataLoader\n    root = Path(data_dir).resolve()\n    labels = root / label_filename\n    digest = file_sha256(labels)\n    if expected_labels_sha256 and digest != expected_labels_sha256:\n        raise ValueError("Evaluation label checksum mismatch")\n    document = json.loads(labels.read_text(encoding="utf-8"))\n    records = document["samples"]\n    if any(row["dataset_split"] != "validation" for row in records):\n        # A main-dataset manifest contains both splits; evaluate validation only.\n        records = [row for row in records if row["dataset_split"] == "validation"]\n    if not records:\n        raise ValueError("No validation records in evaluation manifest")\n    records = training.select_group_subset(records, max_samples, 42)\n    provenance = {"checkpoint_sha256": file_sha256(checkpoint_path), "labels_sha256": digest,\n                  "sample_ids": [row["sample_id"] for row in records],\n                  "runtime_versions": runtime_versions()}\n    out = Path(output_dir)\n    report_path, rows_path = out / "evaluation.json", out / "predictions.csv"\n    reusable_cache = None\n    if report_path.exists() and rows_path.exists():\n        cached = json.loads(report_path.read_text(encoding="utf-8"))\n        if cached.get("provenance") == provenance and cached.get("predictions_sha256") == file_sha256(rows_path):\n            reusable_cache = cached\n        else:\n            raise ValueError("Existing evaluation cache has different provenance; use a new output directory")\n    seen = set()\n    for row in records:\n        path = (root / row["image"]).resolve()\n        if not path.is_relative_to(root) or row["sample_id"] in seen:\n            raise ValueError("Invalid evaluation image path or duplicate sample ID")\n        seen.add(row["sample_id"])\n        if file_sha256(path) != row["image_sha256"]:\n            raise ValueError(f"Evaluation image hash mismatch: {row[\'sample_id\']}")\n        with Image.open(path) as image:\n            if image.mode != "L" or image.size != (row["width"], row["height"]):\n                raise ValueError("Evaluation image dimensions or grayscale mode disagree with labels")\n        for name in ("pivot", "tip"):\n            xy = np.asarray(row["needle_position_px"][name], dtype=float)\n            norm = np.asarray(row["needle_position_normalized"][name], dtype=float)\n            limits = np.array([row["width"]-1, row["height"]-1])\n            if (xy.shape != (2,) or not np.isfinite(xy).all() or not np.all((xy >= 0) & (xy <= limits))\n                    or not np.allclose(xy/limits, norm, atol=1e-6)):\n                raise ValueError("Invalid evaluation keypoint coordinates")\n        if not np.isfinite(float(row.get("rotation_degrees", 0))):\n            raise ValueError("Invalid evaluation rotation angle")\n    if reusable_cache is not None:\n        return reusable_cache\n    model, checkpoint = training.load_model(checkpoint_path, device=device)\n    dataset = training.DashboardDataset(root, records, checkpoint["config"]["input_size"])\n    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)\n    metrics, rows = training.evaluate_model(model, loader, next(model.parameters()).device)\n    out.mkdir(parents=True, exist_ok=True)\n    training._save_csv(rows_path, rows)\n    report = {"provenance": provenance, "checkpoint_path": str(checkpoint_path),\n              "metrics": metrics, "breakdown": rotation_breakdown(training, rows),\n              "predictions_path": str(rows_path), "predictions_sha256": file_sha256(rows_path),\n              "scope": "Synthetic evaluation only; not independent real-camera testing"}\n    report_path.write_text(json.dumps(report, indent=2, allow_nan=False), encoding="utf-8")\n    del model\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    return report\n\n\ndef save_rotation_comparison(training, *, before_predictions, after_predictions, output_path,\n                             evaluation_name):\n    before = read_prediction_rows(before_predictions)\n    after = read_prediction_rows(after_predictions)\n    if {row["sample_id"] for row in before} != {row["sample_id"] for row in after}:\n        raise ValueError("Before/after evaluations must contain the same samples")\n    baseline, selected = rotation_breakdown(training, before), rotation_breakdown(training, after)\n    rows = []\n    for group in baseline:\n        b, a = baseline[group], selected[group]\n        rows.append({"evaluation": evaluation_name, "group": group, "images": a["n"],\n                     "pose_groups": a["pose_groups"], "before_mae_kmh": b["mae_kmh"],\n                     "after_mae_kmh": a["mae_kmh"], "mae_change_kmh": a["mae_kmh"]-b["mae_kmh"],\n                     "before_within_2_percent": b["within_2_kmh_percent"],\n                     "after_within_2_percent": a["within_2_kmh_percent"],\n                     "within_2_change_percentage_points": a["within_2_kmh_percent"]-b["within_2_kmh_percent"],\n                     "before_tip_error_px": b["tip_mae_px"], "after_tip_error_px": a["tip_mae_px"]})\n    training._save_csv(output_path, rows)\n    return rows\n'
EXPECTED_CORE_SHA256 = '9e3a921afda4bb6cdecb58a6b039d3eea5ebed8fca42020983ba43e7c1e9cfc5'
EXPECTED_WORKFLOW_SHA256 = 'e1c97b5ccc8b5947507d2b608fc2f5f3a45fe638fa25673c06273316b52387ef'

def load_embedded(name, filename, source, expected_hash):
    assert hashlib.sha256(source.encode('utf-8')).hexdigest() == expected_hash
    path = WORK_DIR / filename
    path.write_text(source, encoding='utf-8')
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module, path

training, module_path = load_embedded('speedometer_core', 'train_core.py', TRAINING_CORE_SOURCE, EXPECTED_CORE_SHA256)
workflow, workflow_path = load_embedded('speedometer_workflow', 'notebook_workflow.py', WORKFLOW_SOURCE, EXPECTED_WORKFLOW_SHA256)
workflow.ensure_writable(DRIVE_ROOT)
print('Drive write/read check passed; embedded modules loaded.')

## 3. Download and verify the fixed dataset

The public rotation archive downloads only when a verified local copy is unavailable. Archive and label SHA256, individual input hashes, transformed coordinates and group separation are checked. No token or dataset upload is required. The separate unseen-angle set is kept out of training and model selection.

In [ ]:
import tempfile
import requests

def download_dataset(url, destination, expected_sha256, attempts=3):
    destination = Path(destination)
    if destination.is_file() and workflow.file_sha256(destination) == expected_sha256:
        print('Using the verified cached dataset:', destination)
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(1, attempts + 1):
        with tempfile.NamedTemporaryFile(prefix=destination.name + '.', suffix='.part', dir=destination.parent, delete=False) as handle:
            partial = Path(handle.name)
        try:
            print(f'Downloading dataset: attempt {attempt}/{attempts}')
            with requests.get(url, stream=True, timeout=(30,60)) as response:
                response.raise_for_status()
                with partial.open('wb') as target:
                    for block in response.iter_content(1024*1024):
                        if block:
                            target.write(block)
            if workflow.file_sha256(partial) != expected_sha256:
                raise ValueError('Dataset checksum mismatch')
            partial.replace(destination)
            return
        except (requests.RequestException, OSError, ValueError) as error:
            if attempt == attempts:
                raise RuntimeError('Dataset download failed; check the connection and rerun this cell.') from error
            time.sleep(attempt*2)
        finally:
            partial.unlink(missing_ok=True)

download_dataset(DATASET_URL, ZIP_PATH, EXPECTED_ZIP_SHA256)
with zipfile.ZipFile(ZIP_PATH) as archive:
    if archive.testzip() is not None:
        raise ValueError('Dataset archive is damaged')
    for entry in archive.infolist():
        destination = (WORK_DIR / entry.filename).resolve()
        if not destination.is_relative_to(WORK_DIR.resolve()) or not entry.filename.startswith('bw_dataset_rotation_v2/'):
            raise ValueError('Unexpected dataset archive path')
    archive.extractall(WORK_DIR)
DATA_DIR = WORK_DIR / 'bw_dataset_rotation_v2'
ROBUSTNESS_DIR = DATA_DIR / 'robustness'
if workflow.file_sha256(DATA_DIR / 'labels.json') != EXPECTED_LABELS_SHA256:
    raise ValueError('Dataset label provenance mismatch')
records, integrity = training.verify_dataset(DATA_DIR)
assert integrity['split_counts'] == {'train':4800, 'validation':1200}
validation_records = [r for r in records if r['dataset_split'] == 'validation']
assert len({r['group_id'] for r in validation_records}) == 50
if workflow.file_sha256(ROBUSTNESS_DIR / 'labels.json') != EXPECTED_ROBUSTNESS_LABELS_SHA256:
    raise ValueError('Robustness label provenance mismatch')
print('Verified main dataset: 4800 training, 1200 validation, 50 validation poses. Separate robustness:1200 images.')
if (DATA_DIR / 'overview.png').is_file():
    display(DisplayImage(filename=str(DATA_DIR / 'overview.png'), width=700))

## 4. Resolve the experiment and continue training

The saved experiment manifest is authoritative on resume. Default transfer uses 512 pixels, batch size 8, all 4,800 training images, 1,200 validation images and 20 fine-tuning epochs. Fresh experiments use 5 head-training plus 25 fine-tuning epochs. Early stopping uses **only main validation**, and remains enabled. The source model is evaluated on unseen angles before training, and the selected best model is evaluated on exactly those images afterward.

Full checkpoints are staged, copied and rehashed, then published with a `.complete.json` sidecar. Drive needs space for all epoch saves. Incomplete/corrupt files are preserved and skipped if an older verified compatible checkpoint exists. With no valid recovery state, the notebook stops instead of importing legacy weights or starting fresh. The Drive mount cannot guarantee remote upload completion at the instant of a disconnect, so earlier recovery files are retained.

In [ ]:
existing_manifest_path = DRIVE_ROOT / 'experiments' / EXPERIMENT_NAME / 'experiment_config.json'
existing_kind = json.loads(existing_manifest_path.read_text(encoding='utf-8')).get('initialization_kind') if existing_manifest_path.exists() else None
COMPARE_SOURCE_MODEL = TRAIN_MODE != 'fresh' and existing_kind != 'fresh'
evaluation_seed = None
seed_hash = None
if COMPARE_SOURCE_MODEL and RUN_MODE == 'smoke':
    evaluation_seed = WORK_DIR / 'smoke_random_evaluation_seed.pt'
    if not evaluation_seed.exists():
        torch.manual_seed(42)
        random_seed_model = training.SpeedometerModel(pretrained=False)
        torch.save({
            'format_version':1, 'model_state_dict':random_seed_model.state_dict(),
            'config':{'input_size':128,'pretrained':False}, 'best_epoch':0,
            'architecture':training._ARCHITECTURE_ID, 'preprocessing':training._PREPROCESSING_ID,
            'keypoint_order':['pivot','tip'], 'speed_range_kmh':[0,180],
            'labels_sha256':'0'*64, 'validation_metrics':{'mae_kmh':90.0},
            'scope':'Random software-test fixture; no source model performance claim',
        },evaluation_seed)
        del random_seed_model
    seed_hash = workflow.file_sha256(evaluation_seed)
elif COMPARE_SOURCE_MODEL:
    evaluation_seed = SEED_CHECKPOINT
    seed_hash = EXPECTED_SEED_SHA256
    if not evaluation_seed.is_file() or workflow.file_sha256(evaluation_seed) != seed_hash:
        raise ValueError('The source epoch-47 checkpoint is missing or changed; before/after comparison cannot proceed.')
prepared = workflow.prepare_persistent_run(
    training, data_dir=DATA_DIR, drive_root=DRIVE_ROOT, experiment_name=EXPERIMENT_NAME,
    requested_mode=TRAIN_MODE, run_mode=RUN_MODE, fresh_pretrained=FRESH_PRETRAINED,
    import_pilot_if_missing=IMPORT_PILOT_IF_MISSING, warm_start_path=evaluation_seed or SEED_CHECKPOINT,
    extend_by_epochs=EXTEND_BY_EPOCHS, core_sha256=EXPECTED_CORE_SHA256,
    first_import_mode='warm_start_transfer', approved_source_sha256=seed_hash or EXPECTED_SEED_SHA256,
    source_epoch=0 if RUN_MODE=='smoke' else 47, additional_epochs=2 if RUN_MODE=='smoke' else 20, train_sample_limit=None,
    smoke_train_samples=24, smoke_val_samples=24, dataset_name='rotation_v2',
    allow_smoke_transfer=(RUN_MODE=='smoke'),
)
config = prepared['config']
EVALUATIONS_DIR = Path(prepared['experiment_dir']) / 'evaluations'
baseline_robustness = None
if COMPARE_SOURCE_MODEL:
    baseline_robustness = workflow.evaluate_rotation_checkpoint(
        training, checkpoint_path=evaluation_seed, data_dir=ROBUSTNESS_DIR,
        output_dir=EVALUATIONS_DIR / 'source47_unseen_angles',
        expected_labels_sha256=EXPECTED_ROBUSTNESS_LABELS_SHA256,
        batch_size=config.batch_size, max_samples=24 if RUN_MODE=='smoke' else None,
    )
    print('Source model before training — unseen-angle robustness:')
    display(pd.DataFrame(baseline_robustness['breakdown']).T)
else:
    print('Fresh initialization: no private epoch-47 seed is needed. After-training metrics will be reported; no source47 comparison is claimed.')
print('Effective configuration:')
display(pd.Series({key:getattr(config,key) for key in ('input_size','batch_size','head_epochs','finetune_epochs','max_train_samples','max_val_samples','seed')}))
result = training.train_persistent(config, **prepared['kwargs'])
result['run_mode'] = RUN_MODE
SESSION_DIR = Path(prepared['run_dir'])
(SESSION_DIR / 'experiment_results.json').write_text(json.dumps([result], indent=2), encoding='utf-8')
print('Latest complete training state:', result['latest_checkpoint_path'])
print('Best inference weights:', result['checkpoint_path'])
print('Epoch:', result['epochs_run'], '| saved target:', result['target_epoch'], '| epochs this invocation:', result['epochs_this_call'])

## 5. Inspect the saved model

MAE measures average error; **within ±2 km/h** measures the fraction meeting that tolerance. These are different metrics. Inspect worst predictions and tip errors as well as averages. Reports separate lighting/blur profiles, speed ranges and the inferred 60-tick region. No learned confidence or unreadable-frame detector is claimed.

In [ ]:
m = result['metrics']
summary = pd.DataFrame([{
    'mode':prepared['actual_mode'], 'training_images':result['training_samples'],
    'validation_images':result['validation_samples'], 'validation_poses':m['pose_groups'],
    'MAE_kmh':m['mae_kmh'], 'RMSE_kmh':m['rmse_kmh'], 'within_2_kmh_percent':m['within_2_kmh_percent'],
    'P95_error_kmh':m['p95_absolute_error_kmh'], 'tip_error_px':m['tip_mae_px'],
    'best_epoch':result['best_epoch'], 'latest_epoch':result['epochs_run'], 'target_epoch':result['target_epoch'],
    'epochs_this_call':result['epochs_this_call'], 'scope':'synthetic validation' if RUN_MODE=='pilot' else 'software smoke only',
}])
summary.to_csv(SESSION_DIR / 'run_summary.csv', index=False)
display(summary)
display(DisplayImage(filename=result['learning_curves_path']))
display(DisplayImage(filename=result['worst_predictions_path'], width=1100))
display(pd.DataFrame(result['metrics_by_profile']).T)
print('Real-camera accuracy remains unmeasured.' if RUN_MODE=='pilot' else 'SMOKE ONLY — not a trained-model accuracy claim.')

## 6. Compare source epoch 47 with the selected model

Both models see the **same images** in each comparison. Main validation includes angle 0° and the 30° grid used for model selection. The separate unseen-angle robustness set uses the offset 15° grid and is evaluated only before/after training. Its performance never changes the optimizer, early stopping or selected checkpoint. These are synthetic robustness checks, not independent real recordings.

Fresh experiments have no private source-model baseline and report after-training robustness metrics instead.

A negative MAE change means improvement; a positive change in the percentage within ±2 km/h means improvement. Inspect 0° separately to identify any loss of the original dashboard-reading performance.

In [ ]:
main_before_path = result.get('baseline_predictions_path')
if COMPARE_SOURCE_MODEL and main_before_path is None:
    raise RuntimeError('The source-model main-validation baseline is missing; no before/after claim can be made.')
main_comparison = []
if COMPARE_SOURCE_MODEL:
    main_comparison = workflow.save_rotation_comparison(
        training, before_predictions=main_before_path, after_predictions=result['predictions_path'],
        output_path=SESSION_DIR / 'comparison_main_validation.csv', evaluation_name='main_validation',
    )
selected_robustness = workflow.evaluate_rotation_checkpoint(
    training, checkpoint_path=result['checkpoint_path'], data_dir=ROBUSTNESS_DIR,
    output_dir=SESSION_DIR / 'robustness_selected',
    expected_labels_sha256=EXPECTED_ROBUSTNESS_LABELS_SHA256,
    batch_size=config.batch_size, max_samples=24 if RUN_MODE=='smoke' else None,
)
robustness_comparison = []
if COMPARE_SOURCE_MODEL:
    robustness_comparison = workflow.save_rotation_comparison(
        training, before_predictions=baseline_robustness['predictions_path'],
        after_predictions=selected_robustness['predictions_path'],
        output_path=SESSION_DIR / 'comparison_unseen_angles.csv', evaluation_name='unseen_angle_robustness',
    )
comparison = pd.DataFrame(main_comparison + robustness_comparison)
if COMPARE_SOURCE_MODEL:
    comparison.to_csv(SESSION_DIR / 'rotation_comparison.csv',index=False)
    display(comparison)
    shutil.copytree(EVALUATIONS_DIR / 'source47_unseen_angles', SESSION_DIR / 'robustness_source',dirs_exist_ok=True)
    figure,axes=plt.subplots(1,2,figsize=(14,4))
    for axis,rows,title in zip(axes,[main_comparison,robustness_comparison],['Main validation','Unseen-angle robustness']):
        angles=sorted((float(row['group'].removeprefix('angle_')),row) for row in rows if row['group'].startswith('angle_'))
        axis.plot([angle for angle,row in angles],[row['before_mae_kmh'] for angle,row in angles],'o-',label='Source before training')
        axis.plot([angle for angle,row in angles],[row['after_mae_kmh'] for angle,row in angles],'o-',label='Selected after training')
        axis.set(xlabel='Clockwise image rotation (degrees)',ylabel='MAE (km/h)',title=title)
        axis.grid(alpha=0.25)
        axis.legend()
    figure.tight_layout()
    figure.savefig(SESSION_DIR / 'rotation_before_after.png',dpi=140)
    plt.close(figure)
    display(DisplayImage(filename=str(SESSION_DIR / 'rotation_before_after.png')))
else:
    after_only = pd.DataFrame(selected_robustness['breakdown']).T
    after_only.to_csv(SESSION_DIR / 'robustness_after_training.csv')
    display(after_only)
    (SESSION_DIR / 'comparison_availability.json').write_text(json.dumps({
        'source_comparison_available':False,'reason':'Fresh initialization has no private epoch-47 baseline.'
    },indent=2),encoding='utf-8')
    print('Fresh experiment: after-training evaluation only; no before/after improvement claim.')

In [ ]:
saved_model, checkpoint = training.load_model(result['checkpoint_path'])
example = validation_records[0]
prediction = training.predict_image(saved_model, DATA_DIR / example['image'], input_size=checkpoint['config']['input_size'])
with Image.open(DATA_DIR / example['image']) as image:
    pixels = np.asarray(image)
figure, axis = plt.subplots(figsize=(6,6))
axis.imshow(pixels, cmap='gray', vmin=0, vmax=255)
for source,color,style in [(example['needle_position_normalized'],'lime','-'),(prediction['keypoints_normalized'],'magenta','--')]:
    points = np.array([source['pivot'],source['tip']])*511
    axis.plot(points[:,0],points[:,1],style,color=color,marker='o',linewidth=1.5)
axis.set_title(f"Label {example['speed_kmh']:.1f} km/h | prediction {prediction['speed_kmh']:.1f} km/h")
axis.axis('off')
figure.tight_layout()
figure.savefig(SESSION_DIR / 'reloaded_model_example.png',dpi=130)
plt.close(figure)
display(DisplayImage(filename=str(SESSION_DIR / 'reloaded_model_example.png')))

## 7. Export a compact results ZIP

The ZIP contains the best inference model, metrics, predictions, plots, source and a manifest of Drive locations. **Full per-epoch training states remain on Drive** and are excluded from the download to avoid duplicating many large files. Retain the Drive experiment for exact continuation; the compact ZIP alone is an inference/report backup.

After a reset, reopen this notebook, mount the same Drive, retain the experiment name, choose `resume_latest` and Run all. If its fixed budget is complete, no extra epochs run. Use `resume_best` only to rewind to the best complete state. A new training budget requires an explicit positive `EXTEND_BY_EPOCHS`; leave it at 0 on ordinary restarts.

In [ ]:
shutil.copyfile(module_path, SESSION_DIR / 'train_core.py')
shutil.copyfile(workflow_path, SESSION_DIR / 'notebook_workflow.py')
shutil.copyfile(prepared['manifest_path'], SESSION_DIR / 'experiment_config_snapshot.json')
drive_manifest = {
    'drive_root':str(DRIVE_ROOT), 'experiment_dir':prepared['experiment_dir'], 'run_dir':str(SESSION_DIR),
    'experiment_config':prepared['manifest_path'], 'latest_full_checkpoint':result['latest_checkpoint_path'],
    'best_inference_checkpoint':result['checkpoint_path'], 'dataset_labels_sha256':EXPECTED_LABELS_SHA256,
    'robustness_labels_sha256':EXPECTED_ROBUSTNESS_LABELS_SHA256, 'source_checkpoint_sha256':seed_hash,
    'source_epoch':(0 if RUN_MODE=='smoke' else 47) if COMPARE_SOURCE_MODEL else None,
    'training_core_sha256':EXPECTED_CORE_SHA256, 'workflow_sha256':EXPECTED_WORKFLOW_SHA256,
    'target_epoch':result['target_epoch'], 'run_mode':RUN_MODE,
    'scope':'Synthetic development validation; independent real footage still required',
    'archive_excludes':'All epoch_*.pt, epoch_*.complete.json and partial files; retain Drive for full resume',
}
(SESSION_DIR / 'drive_paths.json').write_text(json.dumps(drive_manifest,indent=2),encoding='utf-8')
RESULTS_ZIP = SESSION_DIR.with_name(SESSION_DIR.name+'_results.zip')
with zipfile.ZipFile(RESULTS_ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as archive:
    for path in sorted(SESSION_DIR.rglob('*')):
        if path.is_file() and not path.name.startswith('epoch_') and '.partial' not in path.name:
            archive.write(path,path.relative_to(SESSION_DIR))
print('Compact results archive:',RESULTS_ZIP)
print('Full training states remain under:',prepared['experiment_dir'])
if IN_COLAB and RUN_MODE!='smoke':
    from google.colab import files
    files.download(str(RESULTS_ZIP))